# Visualização Verb <-> Verb (Hiperlink)

In [ ]:
# gerar_visualizacao_final.py
#
# Este script é a etapa final do pipeline, responsável por gerar a
# visualização interativa da rede.
#
# Etapas:
# 1. Carrega o arquivo JSON final, que já contém todas as métricas de rede e as coordenadas (x, y) dos nós.
# 2. Prepara os dados de nós e arestas para a biblioteca Cytoscape.js.
# 3. Utiliza o layout 'preset' do Cytoscape para posicionar os nós
#    exatamente de acordo com as coordenadas pré-calculadas.
# 4. Gera um arquivo HTML autocontido com o grafo interativo, filtros
#    de comunidade/categoria e um painel de detalhes.

import json
import os
from datetime import datetime
from pathlib import Path
import math

def gerar_visualizacao_final():
    # Gera o arquivo HTML da visualização a partir dos dados finais pré-processados.

    # --- 1. CONFIGURAÇÃO DE ARQUIVOS ---
    INPUT_DIR = Path('../dados/dados_com_flags_redirecionamento') 
    INPUT_FILENAME = 'dados_com_constraint_novo.json'
    REDIRECT_MAP_FILENAME = 'redirect_map.json'
    OUTPUT_DIR = Path('../public/Visoes_com_tratamento_redirecionamento')
    OUTPUT_DIR.mkdir(exist_ok=True)
    
    input_path = INPUT_DIR / INPUT_FILENAME
    redirect_map_path = INPUT_DIR / REDIRECT_MAP_FILENAME

    if not input_path.exists():
        print(f"ERRO: O arquivo de dados enriquecidos não foi encontrado em '{input_path}'")
        return
    if not redirect_map_path.exists():
         print(f"AVISO: Arquivo de mapa de redirecionamentos '{redirect_map_path}' não encontrado.")
         print("         As arestas serão criadas sem resolver redirects (podem faltar links).")
         redirect_map = {}
    else:
        print(f"Carregando mapa de redirecionamentos de '{redirect_map_path}'...")
        try:
            with open(redirect_map_path, 'r', encoding='utf-8') as f:
                redirect_map = json.load(f)
            print(f"Mapa carregado com {len(redirect_map)} entradas.")
        except Exception as e:
             print(f"ERRO ao carregar mapa de redirects: {e}. Usando mapa vazio.")
             redirect_map = {}

    # --- 2. CARREGAMENTO DOS DADOS ENRIQUECIDOS ---
    print(f"Carregando dados enriquecidos de '{input_path}'...")
    with open(input_path, 'r', encoding='utf-8') as f:
        dados = json.load(f)

    verbetes_enriquecidos = dados.get('verbetes_completo', [])

    if not verbetes_enriquecidos:
            print("ERRO: Nenhum verbete encontrado ou falha ao carregar.")
            return
    
    id_to_verbete = {str(v['id']): v for v in verbetes_enriquecidos}
    titulos_ids = {v['titulo']: v['id'] for v in verbetes_enriquecidos}
    max_redirect_hops = 5

    # Cria mapa titulo -> id (APENAS para verbetes reais)
    print("Criando mapa Título->ID para nós...")
    titulos_ids = {
        v['titulo']: str(v['id']) 
        for v in verbetes_enriquecidos 
        if 'titulo' in v and 'id' in v
    }
    
    # --- 3. PREPARAÇÃO DOS DADOS PARA O CYTOSCAPE ---
    print("Preparando dados de nós, arestas e filtros para o Cytoscape...")
    
    all_categories = set()
    nodes = []
    for verb in verbetes_enriquecidos:
        # Fórmula para o tamanho do nó, baseada no PageRank
        size = (70 + (verb.get('pagerank', 0) * 80000))

        label_length = len(verb['titulo'])
        if label_length == 0:
            label_length = 1

        SCALE_CONSTANT = 7

        base_font_size = (size / (label_length*2)) * SCALE_CONSTANT
        MIN_BASE_FONT = 8
        MAX_BASE_FONT = 70

        base_font_size = max(MIN_BASE_FONT, min(base_font_size, MAX_BASE_FONT))

        base_text_width = size * 0.9

        # Coleta todas as categorias únicas para o menu de filtro
        clean_cats = []
        if 'categorias' in verb and verb['categorias']:
            for cat in verb['categorias']:
                clean_cat = cat.replace('Categoria:', '').strip()
                if clean_cat:
                    all_categories.add(clean_cat)
                    clean_cats.append(clean_cat)

        nodes.append({
            'data': {
                'id': str(verb['id']),
                'label': verb['titulo'],
                'size': size,
                'color': verb.get('community_color', '#6A737D'),
                'community_id': verb.get('community_id', -1),
                'categories_str': '|' + '|'.join(clean_cats) + '|',
                'pagerank': verb.get('pagerank', 0.0),
                'total_degree': verb.get('total_degree', 0),
                'quantidade_edicoes': verb.get('quantidade_edicoes', 0),
                'betweenness_centrality': verb.get('betweenness_centrality', 0.0),
                'closeness_centrality': verb.get('closeness_centrality', 0.0),
                'constraint': verb.get('constraint', 0.0),
                'metrica_composta': verb.get('metrica_composta', 0.0),
                'base_font_size': base_font_size,
                'base_text_width': base_text_width

            },
            # A posição já vem pronta do arquivo de entrada
            'position': verb.get('position')
        })

    edges = []
    print("Gerando arestas (resolvendo cadeias de redirecionamentos)...")
    arestas_criadas_viz = 0
    refs_resolvidas_viz = 0
    refs_quebradas_viz = 0

    for verb in verbetes_enriquecidos: # Itera sobre os nós de origem
        source_vid = str(verb['id'])
        # Pega a lista de referências que está no JSON
        referencias_originais = verb.get('referencias', []) 

        for ref_titulo in referencias_originais:
            if not isinstance(ref_titulo, str) or not ref_titulo: continue

            current_target_title = ref_titulo
            final_target_title = ref_titulo 
            visited_redirects = {ref_titulo} # Para detectar loops A->B->A
            hops = 0 # Contador de saltos

            # Continua seguindo redirects ENQUANTO o alvo atual AINDA ESTIVER no mapa
            while current_target_title in redirect_map and hops < max_redirect_hops:
                next_target = redirect_map[current_target_title]

                # Detecção de Loop
                if next_target in visited_redirects:
                    final_target_title = current_target_title # Usa o último antes do loop
                    break # Sai do while

                # Atualiza para o próximo salto
                final_target_title = next_target # Guarda o novo alvo
                current_target_title = next_target # Continua a busca a partir dele
                visited_redirects.add(current_target_title)
                hops += 1
                if hops == 1: # Conta apenas uma vez por cadeia resolvida
                    refs_resolvidas_viz += 1

            # Verifica se o título FINAL (após seguir a cadeia) é um verbete real
            if final_target_title in titulos_ids:
                target_vid = titulos_ids[final_target_title]
                # Adiciona a aresta resolvida
                edges.append({'data': {'source': source_vid, 'target': target_vid}})
                arestas_criadas_viz += 1
            else:
                refs_quebradas_viz += 1

    print(f"  - {arestas_criadas_viz} arestas criadas para visualização.")
    print(f"  - {refs_resolvidas_viz} referências que passaram por resolução (1+ saltos).")
    if refs_quebradas_viz > 0:
        print(f"  - {refs_quebradas_viz} referências ignoradas (alvo final inválido ou loop).")

    # Prepara dados para o filtro de comunidades
    community_map = {}
    for verb in verbetes_enriquecidos:
        cid = verb.get('community_id', -1)
        if cid not in community_map:
            community_map[cid] = {'id': cid, 'color': verb.get('community_color', '#6A737D'), 'size': 0}
        community_map[cid]['size'] += 1
    
    if -1 in community_map:
        community_map[-1]['name'] = 'Isolados / Outros'
    
    community_data = list(community_map.values())
    sorted_categories = sorted(list(all_categories))
    
    # Prepara lista de verbetes para o campo de busca
    verbetes_para_busca = sorted(
        [{'value': str(v['id']), 'label': v['titulo']} for v in verbetes_enriquecidos],
        key=lambda x: x['label']
    )

    # --- 4. GERAÇÃO DO CÓDIGO HTML E JAVASCRIPT ---
    id_to_verbete_json = json.dumps(id_to_verbete, ensure_ascii=False)
    nodes_json = json.dumps(nodes, ensure_ascii=False)
    edges_json = json.dumps(edges, ensure_ascii=False)
    community_data_json = json.dumps(community_data, ensure_ascii=False)
    categories_json = json.dumps(sorted_categories, ensure_ascii=False)
    verbetes_para_busca_json = json.dumps(verbetes_para_busca, ensure_ascii=False)
    
    js_template = """
        const verbetes = {id_to_verbete_json};
        const communityData = {community_data_json};
        const categoryData = {categories_json};
        // ALTERAÇÃO 3: Receber os dados dos verbetes para a busca
        const searchData = {verbetes_para_busca_json};

        // Funções auxiliares para formatar os dados no painel de detalhes
        function formatISODate(isoString) {
            if (!isoString || isoString.includes('Não encontrado') || isoString.includes('Erro')) return 'N/A';
            try { return new Date(isoString).toLocaleString('pt-BR', { dateStyle: 'short', timeStyle: 'short' }); } catch (e) { return 'Data inválida'; }
        }
        function formatNumber(num) {
            if (typeof num !== 'number') return 'N/A';
            return num.toFixed(6);
        }
        function createTagList(dataArray, prefix = '') {
            if (!dataArray || dataArray.length === 0) return '<span class="tag-empty">Nenhuma</span>';
            return dataArray.map(item => `<span class="tag">${prefix}${item.trim()}</span>`).join('');
        }
        function createObjectTagList(dataObject) {
            if (!dataObject || Object.keys(dataObject).length === 0) return '<span class="tag-empty">Nenhum</span>';
            return Object.entries(dataObject).map(([key, value]) => `<span class="tag">${key}: ${value}</span>`).join('');
        }

        const layoutOptions = {
            name: 'preset',
            padding: 50,
            fit: true,
        };

        const cy = cytoscape({
          container: document.getElementById('cy'),
          elements: { nodes: {nodes_json}, edges: {edges_json} },

          style: [
            { selector: 'node',
              style: {
                'label': 'data(label)',
                'width': 'data(size)',
                'height': 'data(size)',
                'background-color': 'data(color)',
                'color': '#000000', 
                'font-size': 'data(base_font_size)',
                'text-valign': 'center',
                'text-halign': 'center', 
                'text-wrap': 'wrap', 
                'text-max-width': 'data(base_text_width)',
                'border-color': '#000000', 
                'border-width': '1px', 
                'display': 'element'
            }},
            
            { selector: 'edge',
              style: {
                'width': 1.5, 
                'line-color': '#ffffff', 
                'opacity': 0.3,
                'curve-style': 'bezier', 
                'display': 'element'
            }},
            
            { selector: 'node:selected',
              style: {
                'border-color': '#FFFFFF', 
                'border-width': 15, 
                'color': '#000000', 
            }},

            { selector: '.faded', style: { 'opacity': 0.1, 'text-opacity': 0 } },
          ],
          layout: layoutOptions
        });

        // ALTERAÇÃO 4: Instanciar o Choices.js para o campo de busca
        const searchFilter = new Choices('#search-filter', {
            placeholder: true,
            placeholderValue: 'Buscar verbete específico...',
            allowHTML: false, // Por segurança, não renderizar HTML nas opções de busca
            searchResultLimit: 100,
        });

        const communityFilter = new Choices('#community-filter', {
            removeItemButton: true, 
            placeholder: true, 
            placeholderValue: 'Filtrar por comunidades...', 
            allowHTML: true
        });

        const categoryFilter = new Choices('#category-filter', {
            removeItemButton: true, 
            placeholder: true, 
            placeholderValue: 'Filtrar por categorias...'
        });

        const topNMetricFilter = new Choices('#topn-metric-filter', {
            placeholder: true, 
            placeholderValue: 'Selecionar métrica...',
			allowHTML: false,
			searchEnabled: false,
			itemSelectText: 'Selecionar'
        });

        function populateFilters() {
            // Popula busca
            searchFilter.setChoices(searchData, 'value', 'label', true);

            // Popula filtro de comunidade
            const communityChoices = communityData
                .sort((a, b) => (a.id === -1) - (b.id === -1) || a.id - b.id)
                .map(c => {
                    const label = c.name ? `${c.name} (${c.size})` : `Comunidade ${c.id} (${c.size})`;
                    return { value: c.id, label: `<span class="color-swatch" style="background-color:${c.color};"></span> ${label}` };
                });
            communityFilter.setChoices(communityChoices, 'value', 'label', false);

            // Popula filtro de categoria
            const categoryChoices = categoryData.map(cat => ({ value: cat, label: cat }));
            categoryFilter.setChoices(categoryChoices, 'value', 'label', false);

            const metricChoices = [
				{ value: '', label: 'Nenhuma' },
                { value: 'pagerank', label: 'PageRank' },
                { value: 'total_degree', label: 'Grau Total (Conexões)' },
                { value: 'quantidade_edicoes', label: 'Quantidade de Edições' },
                { value: 'betweenness_centrality', label: 'Betweenness Centrality' },
				{ value: 'closeness_centrality', label: 'Closeness Centrality' },
				{ value: 'constraint', label: 'Constraint' },
                { value: 'metrica_composta', label: 'Métrica Composta' }
            ];
            topNMetricFilter.setChoices(metricChoices, 'value', 'label', false);
        }

        function applyFilters() {
            const selectedCommunities = communityFilter.getValue(true);
            const selectedCategories = categoryFilter.getValue(true);
            const selectedMetric = topNMetricFilter.getValue(true); // Retorna '' se "Nenhuma"
            const topNValue = parseInt(document.getElementById('topn-value-filter').value, 10);

            // --- INÍCIO DA CORREÇÃO LÓGICA ---
            // Define o estado de "nenhum filtro"
            const noCommunityFilter = selectedCommunities.length === 0;
            const noCategoryFilter = selectedCategories.length === 0;
            // O filtro TopN está inativo se a métrica não for selecionada OU o valor não for um número > 0
            const noTopNFilter = !selectedMetric || isNaN(topNValue) || topNValue <= 0;

            // SE NENHUM FILTRO ESTIVER ATIVO, MOSTRA TUDO E PARA.
            if (noCommunityFilter && noCategoryFilter && noTopNFilter) {
                cy.elements().style('display', 'element');
                return; // Para a execução aqui
            }
            // --- FIM DA CORREÇÃO LÓGICA ---


            // Se pelo menos UM filtro estiver ativo, continua a lógica de filtragem:
            
            let nodesToShow = cy.nodes(); // Começa com todos os nós

            // 1. Filtra por Comunidade (se selecionado)
            if (!noCommunityFilter) {
                const communitySelector = selectedCommunities.map(id => `[community_id = ${id}]`).join(', ');
                nodesToShow = nodesToShow.filter(communitySelector);
            }

            // 2. Filtra por Categoria (se selecionado)
            if (!noCategoryFilter) {
                const categorySelector = selectedCategories.map(cat => `[categories_str *= "|${cat}|"]`).join(', ');
                nodesToShow = nodesToShow.filter(categorySelector);
            }
            
            // 3. Filtra por Top N (se selecionado e válido)
            if (!noTopNFilter) {
                // Ordena os nós (apenas os visíveis) pela métrica em ordem decrescente
                nodesToShow = nodesToShow.sort((a, b) => {
                    return b.data(selectedMetric) - a.data(selectedMetric);
                });
                
                // Pega apenas os "N" primeiros
                nodesToShow = nodesToShow.slice(0, topNValue);
            }
            
            // Lógica final de exibição (agora só executa quando há filtros)
            cy.elements().style('display', 'none'); // Esconde tudo
            // Mostra os nós filtrados E as arestas conectadas a eles
            nodesToShow.union(nodesToShow.connectedEdges()).style('display', 'element');
        }
        
        // ALTERAÇÃO 5: Função para buscar, destacar e focar em um verbete
        function searchAndHighlightVerbete() {
            const selectedId = searchFilter.getValue(true);
            if (!selectedId) {
                return;
            }

            const nodeToHighlight = cy.getElementById(selectedId);

            if (nodeToHighlight.length > 0) {
                // Remove o destaque de buscas/cliques anteriores
                cy.elements().removeClass('faded'); 
                
                // Anima o zoom e centraliza a visão no nó encontrado
                cy.animate({
                    zoom: 1.2, // Nível de zoom, ajuste conforme necessário
                    center: { eles: nodeToHighlight },
                    duration: 2000 // Duração da animação em milissegundos
                });

                nodeToHighlight.trigger('tap');
            }
        }

        // --- LÓGICA DE EVENTOS DE FILTRO ---

        // Listener para o botão de busca (inalterado)
        document.getElementById('search-button').addEventListener('click', searchAndHighlightVerbete);

		// Listener para o novo botão "Aplicar Filtros"
        document.getElementById('apply-all-filters').addEventListener('click', applyFilters);

		// Preenche os dropdowns
        populateFilters();

		// Função "Limpar Filtros" atualizada para incluir os novos campos
        document.getElementById('clear-all-filters').addEventListener('click', () => {
            // Limpa dropdowns do Choices.js
            communityFilter.removeActiveItems();
            categoryFilter.removeActiveItems();
            topNMetricFilter.removeActiveItems(); // Limpa métrica Top N
			topNMetricFilter.setChoiceByValue(''); // Reseta para "Nenhuma"
            searchFilter.clearInput();
            searchFilter.removeActiveItems();
			
			// Limpa campo de valor
			document.getElementById('topn-value-filter').value = '';

            applyFilters();
            
            cy.elements().removeClass('faded');
            cy.fit();
            document.getElementById('details').innerHTML = "Clique em um nó para ver detalhes";
        });
        
        cy.on('tap', 'node', function(evt) {
            const node = evt.target;
            const data = verbetes[node.id()];
            const nodeColor = node.data('color'); // Pega a cor do nó clicado
            const connectedEdges = node.neighborhood().edges();

            // 1. Reseta o estilo de TODAS as arestas para o padrão
            cy.edges().style({
                'line-color': '#ffffff',
                'opacity': 0.1,
                'width': 1.5
            });

            // 2. Aplica o novo estilo apenas nas arestas conectadas
            connectedEdges.style({
                'line-color': nodeColor, // Usa a cor do nó
                'opacity': 1.0,
                'width': 4.5
            });
            
            let detailsHTML = `
                <h3>Informações Gerais</h3>
                <div class="info-block"><strong>Título:</strong> ${data.titulo || 'N/A'}</div>
                <div class="info-block"><strong>Link:</strong> <a href="${data.link}" target="_blank">Abrir na Wiki</a></div>
                <div class="info-block"><strong>ID da Comunidade:</strong> ${data.community_id !== -1 ? data.community_id : 'N/A'}</div>
                <div class="info-block"><strong>Autor da Criação:</strong> ${data.autor_criacao || 'N/A'}</div>
                <div class="info-block"><strong>Data de Criação:</strong> ${formatISODate(data.data_criacao)}</div>
                <div class="info-block"><strong>Última Edição:</strong> ${formatISODate(data.data_ultima_edicao)}</div>
                <div class="info-block"><strong>Quantidade de Edições:</strong> ${data.quantidade_edicoes || 0}</div>
                
                <hr>
                <h3>Métricas de Rede</h3>
                <div class="info-block"><strong>PageRank:</strong> ${formatNumber(data.pagerank)}</div>
                <div class="info-block"><strong>Betweenness Centrality:</strong> ${formatNumber(data.betweenness_centrality)}</div>
                <div class="info-block"><strong>Closeness Centrality:</strong> ${formatNumber(data.closeness_centrality)}</div>
                <div class="info-block"><strong>Clustering Coefficient:</strong> ${formatNumber(data.clustering_coefficient)}</div>
                <div class="info-block"><strong>Grau Total (conexões):</strong> ${data.total_degree || 0}</div>
                
                <hr>
                <h3>Dados de Conteúdo</h3>
                <div class="info-block">
                    <strong>Categorias:</strong><br>${createTagList(data.categorias, 'Categoria: ')}
                </div>
                <div class="info-block">
                    <strong>Usuários (edições):</strong><br>${createObjectTagList(data.usuarios_edicoes)}
                </div>
                <div class="info-block">
                    <strong>Referências (links):</strong><br>${createTagList(data.referencias)}
                </div>
            `;


            
            document.getElementById('details').innerHTML = detailsHTML;
            cy.elements().addClass('faded');
            node.neighborhood().union(node).removeClass('faded');
        });

        cy.on('tap', function(evt) {
          if (evt.target === cy) {
            cy.elements().removeClass('faded');
            document.getElementById('details').innerHTML = "Clique em um nó para ver detalhes";
            cy.edges().style({
                'line-color': '#ffffff',
                'opacity': 0.3,
                'width': 1.5
            });
          }
        });
    """

    # Injeta o JSON de busca no template
    js_code = js_template.replace('{id_to_verbete_json}', id_to_verbete_json) \
                         .replace('{nodes_json}', nodes_json) \
                         .replace('{edges_json}', edges_json) \
                         .replace('{community_data_json}', community_data_json) \
                         .replace('{categories_json}', categories_json) \
                         .replace('{verbetes_para_busca_json}', verbetes_para_busca_json)

    # Adiciona o campo de busca e o botão no template HTML
    html_template = f"""
        <!DOCTYPE html><html>
        <head>
        <meta charset="utf-8"><title>Grafo de Referências - Wikifavelas</title>
        <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/choices.js/public/assets/styles/choices.min.css"/>
        <script src="https://cdn.jsdelivr.net/npm/choices.js/public/assets/scripts/choices.min.js"></script>
        <script src="https://unpkg.com/cytoscape@3.24.0/dist/cytoscape.min.js"></script>
        <style>
            body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Helvetica, Arial, sans-serif; margin: 0; display: flex; height: 100vh; background-color: #0d1117; color: #c9d1d9; }}
            #cy {{ position: relative; width: 70%; height: 100%; background-color: #0d1117; }}
            #sidebar {{ width: 30%; padding: 20px; background-color: #161b22; overflow-y: auto; border-left: 1px solid #30363d; box-sizing: border-box; }}
            h2, h3 {{ color: #f0f6fc; border-bottom: 1px solid #30363d; padding-bottom: 10px; margin-top: 0; }}
            h3 {{ padding-top: 10px; padding-bottom: 8px; margin-bottom: 10px; font-size: 1.1em; }}
            hr {{ border: 0; height: 1px; background-color: #30363d; margin: 20px 0; }}
            .controls-container {{ padding-bottom: 15px; margin-bottom: 15px; border-bottom: 1px solid #30363d; }}
            .filter-actions {{ display: flex; justify-content: space-between; margin-top: 10px; }}
            .search-container {{ gap: 8px; margin-bottom: 15px; }}
            #search-filter {{ flex-grow: 1; }}
            .filter-button, #search-button {{
                background-color: #21262d; border: 1px solid #30363d; color: #c9d1d9;
                padding: 5px 10px; border-radius: 6px; cursor: pointer; font-size: 12px;
            }}
            #search-button {{ flex-shrink: 0; }}
            .filter-button:hover, #search-button:hover {{ background-color: #30363d; }}
            .color-swatch {{ width: 12px; height: 12px; border: 1px solid #555; border-radius: 3px; margin-right: 8px; display: inline-block; vertical-align: middle; }}
            .info-block {{ margin-bottom: 8px; font-size: 14px; line-height: 1.5; }}
            .info-block strong {{ color: #8b949e; display: inline-block; width: 180px; vertical-align: top; }}
            .tag {{ display: inline-block; background-color: #21262d; color: #c9d1d9; padding: 4px 10px; margin: 2px; border-radius: 15px; font-size: 12px; border: 1px solid #30363d; }}
            .tag-empty {{ font-style: italic; color: #8b949e; }}
            a {{ color: #58a6ff; text-decoration: none; }}
            a:hover {{ text-decoration: underline; }}
            .choices {{ margin-bottom: 15px; }}
            .choices__inner {{ background-color: #0d1117; border-radius: 6px; border: 1px solid #30363d; padding: 2px 7.5px; min-height: 36px;}}
            .choices[data-type*="select-one"] .choices__inner {{ padding-bottom: 2px; }}
            .choices__list--multiple .choices__item {{ background-color: #0969da; border: 1px solid #30363d; font-size: 12px; }}
            .choices__list--dropdown, .choices__list[aria-expanded] {{ background-color: #FFFFFF; border: 1px solid #30363d;}}
            .choices__list--dropdown .choices__item--selectable {{ color: #000000;}}
            .choices__list--dropdown .choices__item--selectable.is-highlighted {{ background-color: #000000; color: #FFFFFF;}}
            .choices__placeholder {{ color: #8b949e; }}
        </style>
        </head>

        <body>
        <div id="cy"></div>
        <div id="sidebar">
            <div class="controls-container">
                <h3>Controles</h3>
                <div class="search-container">
                    <select id="search-filter"></select>
                    <button id="search-button">Buscar</button>
                </div>
            </div>

            <div class="controls-container">
                <h3>Filtros</h3>
                <div>
                    <label for="community-filter" style="font-size:14px; color:#8b949e; margin-bottom:5px; display:block;">Filtrar por Comunidades:</label>
                    <select id="community-filter" multiple></select>
                </div>
                <div>
                    <label for="category-filter" style="font-size:14px; color:#8b949e; margin-bottom:5px; display:block;">Filtrar por Categorias:</label>
                    <select id="category-filter" multiple></select>
                </div>
                <div style="margin-top: 15px;">
                    <label for="topn-metric-filter" style="font-size:14px; color:#8b949e; margin-bottom:5px; display:block;">Filtrar Top N por Métrica:</label>
                    <select id="topn-metric-filter"></select>
                </div>
                <div style="margin-top: 15px;">
                    <label for="topn-value-filter" style="font-size:14px; color:#8b949e; margin-bottom:5px; display:block;">Valor de N:</label>
                    <input type="number" id="topn-value-filter" placeholder="Ex: 20" min="1" style="width: 100%; box-sizing: border-box; background-color: #0d1117; border-radius: 6px; border: 1px solid #30363d; padding: 7.5px; color: #c9d1d9;">
                </div>
                <div class="filter-actions">
                    <button id="apply-all-filters" class="filter-button" style="background-color: #238636; border-color: #388E4A;">Aplicar Filtros</button>
                    <button id="clear-all-filters" class="filter-button">Limpar Filtros</button>
                </div>
            </div>

            <h2>Detalhes do Verbete</h2>
            <div id="details">Clique em um nó para ver detalhes.</div>
        </div>
        <script>{js_code}</script>
        </body></html>
    """

    # --- 5. SALVANDO O ARQUIVO HTML FINAL ---
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_html_path = OUTPUT_DIR / f'grafo_wikifavelas_final_{ts}.html'
    
    print(f"Salvando visualização final em '{output_html_path}'...")
    with open(output_html_path, 'w', encoding='utf-8') as f:
        f.write(html_template)

    print(f"✅ Visualização gerada com sucesso!")

if __name__ == '__main__':
    gerar_visualizacao_final()

Carregando mapa de redirecionamentos de '..\dados\dados_com_flags_redirecionamento\redirect_map.json'...
Mapa carregado com 748 entradas.
Carregando dados enriquecidos de '..\dados\dados_com_flags_redirecionamento\dados_com_constraint_novo.json'...
Criando mapa Título->ID para nós...
Preparando dados de nós, arestas e filtros para o Cytoscape...
Gerando arestas (resolvendo cadeias de redirecionamentos)...
  - 16071 arestas criadas para visualização.
  - 1576 referências que passaram por resolução (1+ saltos).
  - 536 referências ignoradas (alvo final inválido ou loop).
Salvando visualização final em '..\public\Visoes_com_tratamento_redirecionamento\grafo_wikifavelas_final_20251024_235733.html'...
✅ Visualização gerada com sucesso!


In [2]:
# gerar_visualizacao_betweenness_com_filtros.py
#
# Versão que colore os nós do grafo com base na métrica de 'betweenness_centrality'
# em uma escala contínua, mas MANTÉM os filtros de comunidade e categoria.

import json
import os
from datetime import datetime
from pathlib import Path

# --- FUNÇÃO AUXILIAR PARA O MAPEAMENTO DE CORES ---
def _map_value_to_color(value, min_val, max_val, start_color=(0, 191, 255), end_color=(255, 255, 0)):
    """
    Mapeia um valor numérico para uma cor em um gradiente.
    start_color: azul claro (deepskyblue)
    end_color: amarelo
    """
    if max_val == min_val:
        return f'#{start_color[0]:02x}{start_color[1]:02x}{start_color[2]:02x}'
        
    # Normaliza o valor para o intervalo [0, 1]
    normalized_value = (value - min_val) / (max_val - min_val)
    
    # Interpola linearmente entre as cores de início e fim
    r = int(start_color[0] + normalized_value * (end_color[0] - start_color[0]))
    g = int(start_color[1] + normalized_value * (end_color[1] - start_color[1]))
    b = int(start_color[2] + normalized_value * (end_color[2] - start_color[2]))
    
    return f'#{r:02x}{g:02x}{b:02x}'

def gerar_visualizacao_final():
    """
    Gera o arquivo HTML da visualização a partir dos dados finais pré-processados.
    """

    # --- 1. CONFIGURAÇÃO DE ARQUIVOS ---
    INPUT_DIR = Path('../dados')
    INPUT_FILENAME = 'dados_com_constraint_novo.json'
    OUTPUT_DIR = Path('../public')
    OUTPUT_DIR.mkdir(exist_ok=True)
    
    input_path = INPUT_DIR / INPUT_FILENAME

    if not input_path.exists():
        print(f"ERRO: O arquivo de dados enriquecidos não foi encontrado em '{input_path}'")
        return

    # --- 2. CARREGAMENTO DOS DADOS ENRIQUECIDOS ---
    print(f"Carregando dados enriquecidos de '{input_path}'...")
    with open(input_path, 'r', encoding='utf-8') as f:
        dados = json.load(f)
    
    verbetes_enriquecidos = dados.get('verbetes_completo', [])
    if not verbetes_enriquecidos:
        print("ERRO: Nenhum verbete encontrado no arquivo de entrada.")
        return
        
    id_to_verbete = {str(v['id']): v for v in verbetes_enriquecidos}
    titulos_ids = {v['titulo']: v['id'] for v in verbetes_enriquecidos}
    
    # --- 3. PREPARAÇÃO DOS DADOS PARA O CYTOSCAPE ---
    print("Preparando dados para o Cytoscape com coloração por Betweenness Centrality...")

    # Acha os valores min/max de betweenness para normalizar a cor
    betweenness_values = [v.get('betweenness_centrality', 0) for v in verbetes_enriquecidos]
    min_betweenness = min(betweenness_values)
    max_betweenness = max(betweenness_values)

    all_categories = set()
    nodes = []
    for verb in verbetes_enriquecidos:
        size = (20 + (verb.get('pagerank', 0) * 60000))
        
        # ATUALIZAÇÃO: Calcula a cor baseada na betweenness centrality
        betweenness = verb.get('betweenness_centrality', 0)
        node_color = _map_value_to_color(betweenness, min_betweenness, max_betweenness)

        clean_cats = []
        if 'categorias' in verb and verb['categorias']:
            for cat in verb['categorias']:
                clean_cat = cat.replace('Categoria:', '').strip()
                if clean_cat:
                    all_categories.add(clean_cat)
                    clean_cats.append(clean_cat)

        nodes.append({
            'data': {
                'id': str(verb['id']),
                'label': verb['titulo'],
                'size': size,
                'color': node_color, # Usa a nova cor calculada
                'community_id': verb.get('community_id', -1), # Mantém o ID da comunidade para o filtro
                'categories_str': '|' + '|'.join(clean_cats) + '|'
            },
            'position': verb.get('position')
        })

    edges = []
    for verb in verbetes_enriquecidos:
        source_vid = str(verb['id'])
        for ref_titulo in verb.get('referencias', []):
            if ref_titulo in titulos_ids:
                target_vid = str(titulos_ids[ref_titulo])
                edges.append({'data': {'source': source_vid, 'target': target_vid}})

    # Prepara dados para o filtro de comunidades (mantido)
    community_map = {}
    for verb in verbetes_enriquecidos:
        cid = verb.get('community_id', -1)
        if cid not in community_map:
            # A cor aqui é apenas para a legenda do filtro, não para o grafo
            community_map[cid] = {'id': cid, 'color': verb.get('community_color', '#6A737D'), 'size': 0}
        community_map[cid]['size'] += 1
    
    if -1 in community_map:
        community_map[-1]['name'] = 'Isolados / Outros'
    
    community_data = list(community_map.values())
    sorted_categories = sorted(list(all_categories))

    # --- 4. GERAÇÃO DO CÓDIGO HTML E JAVASCRIPT ---
    id_to_verbete_json = json.dumps(id_to_verbete, ensure_ascii=False)
    nodes_json = json.dumps(nodes, ensure_ascii=False)
    edges_json = json.dumps(edges, ensure_ascii=False)
    community_data_json = json.dumps(community_data, ensure_ascii=False)
    categories_json = json.dumps(sorted_categories, ensure_ascii=False)
    
    html_template = f"""
        <!DOCTYPE html><html>
        <head>
        <meta charset="utf-8"><title>Grafo Wikifavelas - Cor por Betweenness</title>
        <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/choices.js/public/assets/styles/choices.min.css"/>
        <script src="https://cdn.jsdelivr.net/npm/choices.js/public/assets/scripts/choices.min.js"></script>
        <script src="https://unpkg.com/cytoscape@3.24.0/dist/cytoscape.min.js"></script>
        <style>
            body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Helvetica, Arial, sans-serif; margin: 0; display: flex; height: 100vh; background-color: #0d1117; color: #c9d1d9; }}
            #cy {{ position: relative; width: 70%; height: 100%; background-color: #0d1117; }}
            #sidebar {{ width: 30%; padding: 20px; background-color: #161b22; overflow-y: auto; border-left: 1px solid #30363d; box-sizing: border-box; }}
            h2, h3 {{ color: #f0f6fc; border-bottom: 1px solid #30363d; padding-bottom: 10px; margin-top: 0; }}
            h3 {{ padding-top: 10px; padding-bottom: 8px; margin-bottom: 10px; font-size: 1.1em; }}
            hr {{ border: 0; height: 1px; background-color: #30363d; margin: 20px 0; }}
            .controls-container {{ padding-bottom: 15px; margin-bottom: 15px; border-bottom: 1px solid #30363d; }}
            .filter-actions {{ display: flex; justify-content: space-between; margin-top: 10px; }}
            .filter-button {{ background-color: #21262d; border: 1px solid #30363d; color: #c9d1d9; padding: 5px 10px; border-radius: 6px; cursor: pointer; font-size: 12px; }}
            .filter-button:hover {{ background-color: #30363d; }}
            .color-swatch {{ width: 12px; height: 12px; border: 1px solid #555; border-radius: 3px; margin-right: 8px; display: inline-block; vertical-align: middle; }}
            .info-block {{ margin-bottom: 8px; font-size: 14px; line-height: 1.5; }}
            .info-block strong {{ color: #8b949e; display: inline-block; width: 180px; vertical-align: top; }}
            .tag {{ display: inline-block; background-color: #21262d; color: #c9d1d9; padding: 4px 10px; margin: 2px; border-radius: 15px; font-size: 12px; border: 1px solid #30363d; }}
            .tag-empty {{ font-style: italic; color: #8b949e; }}
            a {{ color: #58a6ff; text-decoration: none; }}
            .choices {{ margin-bottom: 15px; }}
            .choices__inner {{ background-color: #0d1117; border-radius: 6px; border: 1px solid #30363d;}}
            .choices__list--multiple .choices__item {{ background-color: #0969da; border-color: #30363d; }}
            .choices__list--dropdown {{ background-color: #161b22; border-color: #30363d; }}
            .choices__placeholder {{ color: #8b949e; }}
        </style>
        </head>
        <body>
        <div id="cy"></div>
        <div id="sidebar">
            <h2>Detalhes do Verbete</h2>
            <div id="details">Clique em um nó para ver detalhes.</div>
            <div class="controls-container">
                <h3>Filtros</h3>
                <div>
                    <label for="community-filter" style="font-size:14px; color:#8b949e; margin-bottom:5px; display:block;">Filtrar por Comunidades:</label>
                    <select id="community-filter" multiple></select>
                </div>
                <div>
                    <label for="category-filter" style="font-size:14px; color:#8b949e; margin-bottom:5px; display:block;">Filtrar por Categorias:</label>
                    <select id="category-filter" multiple></select>
                </div>
                <div class="filter-actions">
                    <button id="select-all-communities" class="filter-button">Selecionar Todas</button>
                    <button id="clear-all-filters" class="filter-button">Limpar Filtros</button>
                </div>
            </div>
        </div>
        <script>
            const verbetes = {id_to_verbete_json};
            const communityData = {community_data_json};
            const categoryData = {categories_json};

            function formatISODate(isoString) {{
                if (!isoString || isoString.includes('Não encontrado') || isoString.includes('Erro')) return 'N/A';
                try {{ return new Date(isoString).toLocaleString('pt-BR', {{ dateStyle: 'short', timeStyle: 'short' }}); }} catch (e) {{ return 'Data inválida'; }}
            }}
            function formatNumber(num) {{
                if (typeof num !== 'number') return 'N/A';
                return num.toFixed(6);
            }}
            function createTagList(dataArray, prefix = '') {{
                if (!dataArray || dataArray.length === 0) return '<span class="tag-empty">Nenhuma</span>';
                return dataArray.map(item => `<span class="tag">${{prefix}}${{item.trim()}}</span>`).join('');
            }}
            function createObjectTagList(dataObject) {{
                if (!dataObject || Object.keys(dataObject).length === 0) return '<span class="tag-empty">Nenhum</span>';
                return Object.entries(dataObject).map(([key, value]) => `<span class="tag">${{key}}: ${{value}}</span>`).join('');
            }}

            const layoutOptions = {{ name: 'preset', padding: 50, fit: true }};

            const cy = cytoscape({{
              container: document.getElementById('cy'),
              elements: {{ nodes: {nodes_json}, edges: {edges_json} }},
              style: [
                {{ selector: 'node', style: {{
                    'label': 'data(label)', 'width': 'data(size)', 'height': 'data(size)',
                    'background-color': 'data(color)', 'color': '#000000', 
                    'font-size': '12px', 'text-valign': 'center', 'text-halign': 'center', 
                    'text-wrap': 'wrap', 'text-max-width': '100px',
                    'border-color': '#000000', 'border-width': '1px', 'display': 'element'
                }}}},
                {{ selector: 'edge', style: {{
                    'width': 1.5, 'line-color': '#ffffff', 'opacity': 0.3,
                    'curve-style': 'bezier', 'display': 'element'
                }}}},
                {{ selector: 'node:selected', style: {{
                    'border-color': '#FFFF00', 'border-width': 6, 'color': '#000000',
                    'text-outline-color': '#FFFF00', 'text-outline-width': 1
                }}}},
                {{ selector: '.faded', style: {{ 'opacity': 0.1, 'text-opacity': 0 }} }}
              ],
              layout: layoutOptions
            }});

            const communityFilter = new Choices('#community-filter', {{
                removeItemButton: true, placeholder: true, placeholderValue: 'Filtrar por comunidades...', allowHTML: true
            }});
            const categoryFilter = new Choices('#category-filter', {{
                removeItemButton: true, placeholder: true, placeholderValue: 'Filtrar por categorias...'
            }});

            function populateFilters() {{
                const communityChoices = communityData
                    .sort((a, b) => (a.id === -1) - (b.id === -1) || a.id - b.id)
                    .map(c => {{
                        const label = c.name ? `${{c.name}} (${{c.size}})` : `Comunidade ${{c.id}} (${{c.size}})`;
                        return {{ value: c.id, label: `<span class="color-swatch" style="background-color:${{c.color}};"></span> ${{label}}` }};
                    }});
                communityFilter.setChoices(communityChoices, 'value', 'label', false);

                const categoryChoices = categoryData.map(cat => ({{ value: cat, label: cat }}));
                categoryFilter.setChoices(categoryChoices, 'value', 'label', false);
            }}

            function applyFilters() {{
                const selectedCommunities = communityFilter.getValue(true);
                const selectedCategories = categoryFilter.getValue(true);

                if (selectedCommunities.length === 0 && selectedCategories.length === 0) {{
                    cy.elements().style('display', 'element');
                    return;
                }}
                let nodesToShow = cy.nodes();
                if (selectedCommunities.length > 0) {{
                    const communitySelector = selectedCommunities.map(id => `[community_id = ${{id}}]`).join(', ');
                    nodesToShow = nodesToShow.filter(communitySelector);
                }}
                if (selectedCategories.length > 0) {{
                    const categorySelector = selectedCategories.map(cat => `[categories_str *= "|${{cat}}|"]`).join(', ');
                    nodesToShow = nodesToShow.filter(categorySelector);
                }}
                cy.elements().style('display', 'none');
                nodesToShow.union(nodesToShow.connectedEdges()).style('display', 'element');
            }}

            document.getElementById('community-filter').addEventListener('change', applyFilters);
            document.getElementById('category-filter').addEventListener('change', applyFilters);
            populateFilters();

            document.getElementById('select-all-communities').addEventListener('click', () => {{
                const allCommunityValues = communityData.map(c => String(c.id)); 
                communityFilter.setValue(allCommunityValues);
            }});
            document.getElementById('clear-all-filters').addEventListener('click', () => {{
                communityFilter.removeActiveItems();
                categoryFilter.removeActiveItems();
            }});
            
            cy.on('tap', 'node', function(evt) {{
                const node = evt.target;
                const data = verbetes[node.id()];
                let detailsHTML = `
                    <h3>Informações Gerais</h3>
                    <div class="info-block"><strong>Título:</strong> ${{data.titulo || 'N/A'}}</div>
                    <div class="info-block"><strong>Link:</strong> <a href="${{data.link}}" target="_blank">Abrir na Wiki</a></div>
                    <div class="info-block"><strong>ID da Comunidade:</strong> ${{data.community_id !== -1 ? data.community_id : 'N/A'}}</div>
                    <hr>
                    <h3>Métricas de Rede</h3>
                    <div class="info-block"><strong>PageRank:</strong> ${{formatNumber(data.pagerank)}}</div>
                    <div class="info-block"><strong>Betweenness Centrality:</strong> ${{formatNumber(data.betweenness_centrality)}}</div>
                    <hr>
                    <h3>Dados de Conteúdo</h3>
                    <div class="info-block"><strong>Categorias:</strong><br>${{createTagList(data.categorias, 'Cat: ')}}</div>
                `;
                document.getElementById('details').innerHTML = detailsHTML;
                cy.elements().addClass('faded');
                node.neighborhood().union(node).removeClass('faded');
            }});

            cy.on('tap', function(evt) {{
              if (evt.target === cy) {{
                cy.elements().removeClass('faded');
                document.getElementById('details').innerHTML = "Clique em um nó para ver detalhes";
              }}
            }});
        </script>
        </body></html>
    """

    # --- 5. SALVANDO O ARQUIVO HTML FINAL ---
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_html_path = OUTPUT_DIR / f'grafo_betweenness_{ts}.html'
    
    with open(output_html_path, 'w', encoding='utf-8') as f:
        f.write(html_template)

    print(f"✅ Visualização gerada com sucesso em '{output_html_path}'!")

if __name__ == '__main__':
    gerar_visualizacao_final()


Carregando dados enriquecidos de '..\dados\dados_com_constraint_novo.json'...
Preparando dados para o Cytoscape com coloração por Betweenness Centrality...
✅ Visualização gerada com sucesso em '..\public\grafo_betweenness_20251016_024955.html'!


In [3]:
# gerar_visualizacao_closeness_percentil.py
#
# Versão que colore os nós do grafo com base em PERCENTIS da métrica 'closeness_centrality',
# usando uma escala de cores categórica para destacar os nós mais importantes.

import json
import os
from datetime import datetime
from pathlib import Path
import math

def gerar_visualizacao_por_closeness_percentil():
    """
    Gera o arquivo HTML da visualização com cores baseadas em percentis
    da centralidade de proximidade.
    """

    # --- 1. CONFIGURAÇÃO DE ARQUIVOS ---
    INPUT_DIR = Path('../dados')
    INPUT_FILENAME = 'dados_com_constraint_novo.json'
    OUTPUT_DIR = Path('../public')
    OUTPUT_DIR.mkdir(exist_ok=True)
    
    input_path = INPUT_DIR / INPUT_FILENAME

    if not input_path.exists():
        print(f"ERRO: O arquivo de dados enriquecidos não foi encontrado em '{input_path}'")
        return

    # --- 2. CARREGAMENTO DOS DADOS ENRIQUECIDOS ---
    print(f"Carregando dados enriquecidos de '{input_path}'...")
    with open(input_path, 'r', encoding='utf-8') as f:
        dados = json.load(f)
    
    verbetes_enriquecidos = dados.get('verbetes_completo', [])
    if not verbetes_enriquecidos:
        print("ERRO: Nenhum verbete encontrado no arquivo de entrada.")
        return
        
    id_to_verbete = {str(v['id']): v for v in verbetes_enriquecidos}
    titulos_ids = {v['titulo']: v['id'] for v in verbetes_enriquecidos}
    
    # --- 3. PREPARAÇÃO DOS DADOS PARA O CYTOSCAPE ---
    print("Preparando dados para o Cytoscape com coloração por percentil de Closeness...")

    # --- ATUALIZAÇÃO: Lógica de Escala por Percentil ---
    # 1. Ordena os verbetes por closeness_centrality (do maior para o menor)
    verbetes_ordenados = sorted(
        verbetes_enriquecidos, 
        key=lambda v: v.get('closeness_centrality', 0), 
        reverse=True
    )
    
    # 2. Calcula os pontos de corte dos percentis
    total_verbetes = len(verbetes_ordenados)
    idx_top_1_percent = int(total_verbetes * 0.01)
    idx_top_10_percent = int(total_verbetes * 0.10)
    idx_top_50_percent = int(total_verbetes * 0.50)

    # 3. Cria um mapa de ID para cor com base na posição ordenada
    id_to_color = {}
    for i, verbete in enumerate(verbetes_ordenados):
        verbete_id = str(verbete['id'])
        if i < idx_top_1_percent:
            id_to_color[verbete_id] = '#FF0000'  # Vermelho
        elif i < idx_top_10_percent:
            id_to_color[verbete_id] = '#FFFF00'  # Amarelo
        elif i < idx_top_50_percent:
            id_to_color[verbete_id] = '#FFFFFF'  # Branco
        else:
            id_to_color[verbete_id] = '#00B1EC'  # Azul Claro

    all_categories = set()
    nodes = []
    for verb in verbetes_enriquecidos:
        size = (20 + (verb.get('pagerank', 0) * 60000))
        verb_id = str(verb['id'])
        
        # ATUALIZAÇÃO: Atribui a cor a partir do mapa de percentis
        node_color = id_to_color.get(verb_id, '#808080') # Cinza como fallback

        clean_cats = []
        if 'categorias' in verb and verb['categorias']:
            for cat in verb['categorias']:
                clean_cat = cat.replace('Categoria:', '').strip()
                if clean_cat:
                    all_categories.add(clean_cat)
                    clean_cats.append(clean_cat)

        nodes.append({
            'data': {
                'id': verb_id,
                'label': verb['titulo'],
                'size': size,
                'color': node_color, # Usa a nova cor categórica
                'community_id': verb.get('community_id', -1),
                'categories_str': '|' + '|'.join(clean_cats) + '|'
            },
            'position': verb.get('position')
        })

    edges = []
    for verb in verbetes_enriquecidos:
        source_vid = str(verb['id'])
        for ref_titulo in verb.get('referencias', []):
            if ref_titulo in titulos_ids:
                target_vid = str(titulos_ids[ref_titulo])
                edges.append({'data': {'source': source_vid, 'target': target_vid}})

    community_map = {}
    for verb in verbetes_enriquecidos:
        cid = verb.get('community_id', -1)
        if cid not in community_map:
            community_map[cid] = {'id': cid, 'color': verb.get('community_color', '#6A737D'), 'size': 0}
        community_map[cid]['size'] += 1
    
    if -1 in community_map:
        community_map[-1]['name'] = 'Isolados / Outros'
    
    community_data = list(community_map.values())
    sorted_categories = sorted(list(all_categories))

    # --- 4. GERAÇÃO DO CÓDIGO HTML E JAVASCRIPT ---
    id_to_verbete_json = json.dumps(id_to_verbete, ensure_ascii=False)
    nodes_json = json.dumps(nodes, ensure_ascii=False)
    edges_json = json.dumps(edges, ensure_ascii=False)
    community_data_json = json.dumps(community_data, ensure_ascii=False)
    categories_json = json.dumps(sorted_categories, ensure_ascii=False)
    
    html_template = f"""
        <!DOCTYPE html><html>
        <head>
        <meta charset="utf-8"><title>Grafo Wikifavelas - Cor por Percentil de Closeness</title>
        <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/choices.js/public/assets/styles/choices.min.css"/>
        <script src="https://cdn.jsdelivr.net/npm/choices.js/public/assets/scripts/choices.min.js"></script>
        <script src="https://unpkg.com/cytoscape@3.24.0/dist/cytoscape.min.js"></script>
        <style>
            body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Helvetica, Arial, sans-serif; margin: 0; display: flex; height: 100vh; background-color: #0d1117; color: #c9d1d9; }}
            #cy {{ position: relative; width: 70%; height: 100%; background-color: #0d1117; }}
            #sidebar {{ width: 30%; padding: 20px; background-color: #161b22; overflow-y: auto; border-left: 1px solid #30363d; box-sizing: border-box; }}
            h2, h3 {{ color: #f0f6fc; border-bottom: 1px solid #30363d; padding-bottom: 10px; margin-top: 0; }}
            h3 {{ padding-top: 10px; padding-bottom: 8px; margin-bottom: 10px; font-size: 1.1em; }}
            hr {{ border: 0; height: 1px; background-color: #30363d; margin: 20px 0; }}
            .controls-container {{ padding-bottom: 15px; margin-bottom: 15px; border-bottom: 1px solid #30363d; }}
            .filter-actions {{ display: flex; justify-content: space-between; margin-top: 10px; }}
            .filter-button {{ background-color: #21262d; border: 1px solid #30363d; color: #c9d1d9; padding: 5px 10px; border-radius: 6px; cursor: pointer; font-size: 12px; }}
            .filter-button:hover {{ background-color: #30363d; }}
            .color-swatch {{ width: 12px; height: 12px; border: 1px solid #555; border-radius: 3px; margin-right: 8px; display: inline-block; vertical-align: middle; }}
            .info-block {{ margin-bottom: 8px; font-size: 14px; line-height: 1.5; }}
            .info-block strong {{ color: #8b949e; display: inline-block; width: 180px; vertical-align: top; }}
            .tag {{ display: inline-block; background-color: #21262d; color: #c9d1d9; padding: 4px 10px; margin: 2px; border-radius: 15px; font-size: 12px; border: 1px solid #30363d; }}
            .tag-empty {{ font-style: italic; color: #8b949e; }}
            a {{ color: #58a6ff; text-decoration: none; }}
            .choices {{ margin-bottom: 15px; }}
            .choices__inner {{ background-color: #0d1117; border-radius: 6px; border: 1px solid #30363d;}}
            .choices__list--multiple .choices__item {{ background-color: #0969da; border-color: #30363d; }}
            .choices__list--dropdown {{ background-color: #161b22; border-color: #30363d; }}
            .choices__placeholder {{ color: #8b949e; }}
        </style>
        </head>
        <body>
        <div id="cy"></div>
        <div id="sidebar">
            <h2>Detalhes do Verbete</h2>
            <div id="details">Clique em um nó para ver detalhes.</div>
            <div class="controls-container">
                <h3>Filtros</h3>
                <div>
                    <label for="community-filter" style="font-size:14px; color:#8b949e; margin-bottom:5px; display:block;">Filtrar por Comunidades:</label>
                    <select id="community-filter" multiple></select>
                </div>
                <div>
                    <label for="category-filter" style="font-size:14px; color:#8b949e; margin-bottom:5px; display:block;">Filtrar por Categorias:</label>
                    <select id="category-filter" multiple></select>
                </div>
                <div class="filter-actions">
                    <button id="select-all-communities" class="filter-button">Selecionar Todas</button>
                    <button id="clear-all-filters" class="filter-button">Limpar Filtros</button>
                </div>
            </div>
        </div>
        <script>
            const verbetes = {id_to_verbete_json};
            const communityData = {community_data_json};
            const categoryData = {categories_json};

            function formatISODate(isoString) {{
                if (!isoString || isoString.includes('Não encontrado') || isoString.includes('Erro')) return 'N/A';
                try {{ return new Date(isoString).toLocaleString('pt-BR', {{ dateStyle: 'short', timeStyle: 'short' }}); }} catch (e) {{ return 'Data inválida'; }}
            }}
            function formatNumber(num) {{
                if (typeof num !== 'number') return 'N/A';
                return num.toFixed(6);
            }}
            function createTagList(dataArray, prefix = '') {{
                if (!dataArray || dataArray.length === 0) return '<span class="tag-empty">Nenhuma</span>';
                return dataArray.map(item => `<span class="tag">${{prefix}}${{item.trim()}}</span>`).join('');
            }}
            function createObjectTagList(dataObject) {{
                if (!dataObject || Object.keys(dataObject).length === 0) return '<span class="tag-empty">Nenhum</span>';
                return Object.entries(dataObject).map(([key, value]) => `<span class="tag">${{key}}: ${{value}}</span>`).join('');
            }}

            const layoutOptions = {{ name: 'preset', padding: 50, fit: true }};

            const cy = cytoscape({{
              container: document.getElementById('cy'),
              elements: {{ nodes: {nodes_json}, edges: {edges_json} }},
              style: [
                {{ selector: 'node', style: {{
                    'label': 'data(label)', 'width': 'data(size)', 'height': 'data(size)',
                    'background-color': 'data(color)', 'color': '#000000', 
                    'font-size': '12px', 'text-valign': 'center', 'text-halign': 'center', 
                    'text-wrap': 'wrap', 'text-max-width': '100px',
                    'border-color': '#000000', 'border-width': '1px', 'display': 'element'
                }}}},
                {{ selector: 'edge', style: {{
                    'width': 1.5, 'line-color': '#ffffff', 'opacity': 0.3,
                    'curve-style': 'bezier', 'display': 'element'
                }}}},
                {{ selector: 'node:selected', style: {{
                    'border-color': '#FFFF00', 'border-width': 6, 'color': '#000000',
                    'text-outline-color': '#FFFF00', 'text-outline-width': 1
                }}}},
                {{ selector: '.faded', style: {{ 'opacity': 0.1, 'text-opacity': 0 }} }}
              ],
              layout: layoutOptions
            }});

            const communityFilter = new Choices('#community-filter', {{
                removeItemButton: true, placeholder: true, placeholderValue: 'Filtrar por comunidades...', allowHTML: true
            }});
            const categoryFilter = new Choices('#category-filter', {{
                removeItemButton: true, placeholder: true, placeholderValue: 'Filtrar por categorias...'
            }});

            function populateFilters() {{
                const communityChoices = communityData
                    .sort((a, b) => (a.id === -1) - (b.id === -1) || a.id - b.id)
                    .map(c => {{
                        const label = c.name ? `${{c.name}} (${{c.size}})` : `Comunidade ${{c.id}} (${{c.size}})`;
                        return {{ value: c.id, label: `<span class="color-swatch" style="background-color:${{c.color}};"></span> ${{label}}` }};
                    }});
                communityFilter.setChoices(communityChoices, 'value', 'label', false);

                const categoryChoices = categoryData.map(cat => ({{ value: cat, label: cat }}));
                categoryFilter.setChoices(categoryChoices, 'value', 'label', false);
            }}

            function applyFilters() {{
                const selectedCommunities = communityFilter.getValue(true);
                const selectedCategories = categoryFilter.getValue(true);

                if (selectedCommunities.length === 0 && selectedCategories.length === 0) {{
                    cy.elements().style('display', 'element');
                    return;
                }}
                let nodesToShow = cy.nodes();
                if (selectedCommunities.length > 0) {{
                    const communitySelector = selectedCommunities.map(id => `[community_id = ${{id}}]`).join(', ');
                    nodesToShow = nodesToShow.filter(communitySelector);
                }}
                if (selectedCategories.length > 0) {{
                    const categorySelector = selectedCategories.map(cat => `[categories_str *= "|${{cat}}|"]`).join(', ');
                    nodesToShow = nodesToShow.filter(categorySelector);
                }}
                cy.elements().style('display', 'none');
                nodesToShow.union(nodesToShow.connectedEdges()).style('display', 'element');
            }}

            document.getElementById('community-filter').addEventListener('change', applyFilters);
            document.getElementById('category-filter').addEventListener('change', applyFilters);
            populateFilters();

            document.getElementById('select-all-communities').addEventListener('click', () => {{
                const allCommunityValues = communityData.map(c => String(c.id)); 
                communityFilter.setValue(allCommunityValues);
            }});
            document.getElementById('clear-all-filters').addEventListener('click', () => {{
                communityFilter.removeActiveItems();
                categoryFilter.removeActiveItems();
            }});
            
            cy.on('tap', 'node', function(evt) {{
                const node = evt.target;
                const data = verbetes[node.id()];
                let detailsHTML = `
                    <h3>Informações Gerais</h3>
                    <div class="info-block"><strong>Título:</strong> ${{data.titulo || 'N/A'}}</div>
                    <div class="info-block"><strong>Link:</strong> <a href="${{data.link}}" target="_blank">Abrir na Wiki</a></div>
                    <div class="info-block"><strong>ID da Comunidade:</strong> ${{data.community_id !== -1 ? data.community_id : 'N/A'}}</div>
                    <hr>
                    <h3>Métricas de Rede</h3>
                    <div class="info-block"><strong>PageRank:</strong> ${{formatNumber(data.pagerank)}}</div>
                    <div class="info-block"><strong>Betweenness Centrality:</strong> ${{formatNumber(data.betweenness_centrality)}}</div>
                    <div class="info-block"><strong>Closeness Centrality:</strong> ${{formatNumber(data.closeness_centrality)}}</div>
                    <hr>
                    <h3>Dados de Conteúdo</h3>
                    <div class="info-block"><strong>Categorias:</strong><br>${{createTagList(data.categorias, 'Cat: ')}}</div>
                `;
                document.getElementById('details').innerHTML = detailsHTML;
                cy.elements().addClass('faded');
                node.neighborhood().union(node).removeClass('faded');
            }});

            cy.on('tap', function(evt) {{
              if (evt.target === cy) {{
                cy.elements().removeClass('faded');
                document.getElementById('details').innerHTML = "Clique em um nó para ver detalhes";
              }}
            }});
        </script>
        </body></html>
    """

    # --- 5. SALVANDO O ARQUIVO HTML FINAL ---
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_html_path = OUTPUT_DIR / f'grafo_closeness_percentil_{ts}.html'
    
    with open(output_html_path, 'w', encoding='utf-8') as f:
        f.write(html_template)

    print(f"✅ Visualização gerada com sucesso em '{output_html_path}'!")

if __name__ == '__main__':
    gerar_visualizacao_por_closeness_percentil()


Carregando dados enriquecidos de '..\dados\dados_com_constraint_novo.json'...
Preparando dados para o Cytoscape com coloração por percentil de Closeness...
✅ Visualização gerada com sucesso em '..\public\grafo_closeness_percentil_20251016_025005.html'!


In [4]:
# gerar_visualizacao_degree_percentil.py
#
# Versão que colore os nós do grafo com base em PERCENTIS da métrica 'total_degree',
# usando uma escala de cores categórica para destacar os nós mais conectados (hubs).

import json
import os
from datetime import datetime
from pathlib import Path
import math

def gerar_visualizacao_por_degree_percentil():
    """
    Gera o arquivo HTML da visualização com cores baseadas em percentis
    da centralidade de grau (total_degree).
    """

    # --- 1. CONFIGURAÇÃO DE ARQUIVOS ---
    INPUT_DIR = Path('../dados')
    INPUT_FILENAME = 'dados_com_constraint_novo.json'
    OUTPUT_DIR = Path('../public')
    OUTPUT_DIR.mkdir(exist_ok=True)
    
    input_path = INPUT_DIR / INPUT_FILENAME

    if not input_path.exists():
        print(f"ERRO: O arquivo de dados enriquecidos não foi encontrado em '{input_path}'")
        return

    # --- 2. CARREGAMENTO DOS DADOS ENRIQUECIDOS ---
    print(f"Carregando dados enriquecidos de '{input_path}'...")
    with open(input_path, 'r', encoding='utf-8') as f:
        dados = json.load(f)
    
    verbetes_enriquecidos = dados.get('verbetes_completo', [])
    if not verbetes_enriquecidos:
        print("ERRO: Nenhum verbete encontrado no arquivo de entrada.")
        return
        
    id_to_verbete = {str(v['id']): v for v in verbetes_enriquecidos}
    titulos_ids = {v['titulo']: v['id'] for v in verbetes_enriquecidos}
    
    # --- 3. PREPARAÇÃO DOS DADOS PARA O CYTOSCAPE ---
    print("Preparando dados para o Cytoscape com coloração por percentil de Degree Centrality...")

    # --- ATUALIZAÇÃO: Lógica de Escala por Percentil para 'total_degree' ---
    # 1. Ordena os verbetes por total_degree (do maior para o menor)
    verbetes_ordenados = sorted(
        verbetes_enriquecidos, 
        key=lambda v: v.get('total_degree', 0), 
        reverse=True
    )
    
    # 2. Calcula os pontos de corte dos percentis
    total_verbetes = len(verbetes_ordenados)
    idx_top_1_percent = int(total_verbetes * 0.01)
    idx_top_10_percent = int(total_verbetes * 0.10)
    idx_top_50_percent = int(total_verbetes * 0.50)

    # 3. Cria um mapa de ID para cor com base na posição ordenada
    id_to_color = {}
    for i, verbete in enumerate(verbetes_ordenados):
        verbete_id = str(verbete['id'])
        if i < idx_top_1_percent:
            id_to_color[verbete_id] = '#FF0000'  # Vermelho
        elif i < idx_top_10_percent:
            id_to_color[verbete_id] = '#FFFF00'  # Amarelo
        elif i < idx_top_50_percent:
            id_to_color[verbete_id] = '#FFFFFF'  # Branco
        else:
            id_to_color[verbete_id] = '#00B1EC'  # Azul Claro

    all_categories = set()
    nodes = []
    for verb in verbetes_enriquecidos:
        size = (20 + (verb.get('pagerank', 0) * 60000))
        verb_id = str(verb['id'])
        
        # ATUALIZAÇÃO: Atribui a cor a partir do mapa de percentis
        node_color = id_to_color.get(verb_id, '#808080') # Cinza como fallback

        clean_cats = []
        if 'categorias' in verb and verb['categorias']:
            for cat in verb['categorias']:
                clean_cat = cat.replace('Categoria:', '').strip()
                if clean_cat:
                    all_categories.add(clean_cat)
                    clean_cats.append(clean_cat)

        nodes.append({
            'data': {
                'id': verb_id,
                'label': verb['titulo'],
                'size': size,
                'color': node_color, # Usa a nova cor categórica
                'community_id': verb.get('community_id', -1),
                'categories_str': '|' + '|'.join(clean_cats) + '|'
            },
            'position': verb.get('position')
        })

    edges = []
    for verb in verbetes_enriquecidos:
        source_vid = str(verb['id'])
        for ref_titulo in verb.get('referencias', []):
            if ref_titulo in titulos_ids:
                target_vid = str(titulos_ids[ref_titulo])
                edges.append({'data': {'source': source_vid, 'target': target_vid}})

    community_map = {}
    for verb in verbetes_enriquecidos:
        cid = verb.get('community_id', -1)
        if cid not in community_map:
            community_map[cid] = {'id': cid, 'color': verb.get('community_color', '#6A737D'), 'size': 0}
        community_map[cid]['size'] += 1
    
    if -1 in community_map:
        community_map[-1]['name'] = 'Isolados / Outros'
    
    community_data = list(community_map.values())
    sorted_categories = sorted(list(all_categories))

    # --- 4. GERAÇÃO DO CÓDIGO HTML E JAVASCRIPT ---
    id_to_verbete_json = json.dumps(id_to_verbete, ensure_ascii=False)
    nodes_json = json.dumps(nodes, ensure_ascii=False)
    edges_json = json.dumps(edges, ensure_ascii=False)
    community_data_json = json.dumps(community_data, ensure_ascii=False)
    categories_json = json.dumps(sorted_categories, ensure_ascii=False)
    
    html_template = f"""
        <!DOCTYPE html><html>
        <head>
        <meta charset="utf-8"><title>Grafo Wikifavelas - Cor por Percentil de Degree</title>
        <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/choices.js/public/assets/styles/choices.min.css"/>
        <script src="https://cdn.jsdelivr.net/npm/choices.js/public/assets/scripts/choices.min.js"></script>
        <script src="https://unpkg.com/cytoscape@3.24.0/dist/cytoscape.min.js"></script>
        <style>
            body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Helvetica, Arial, sans-serif; margin: 0; display: flex; height: 100vh; background-color: #0d1117; color: #c9d1d9; }}
            #cy {{ position: relative; width: 70%; height: 100%; background-color: #0d1117; }}
            #sidebar {{ width: 30%; padding: 20px; background-color: #161b22; overflow-y: auto; border-left: 1px solid #30363d; box-sizing: border-box; }}
            h2, h3 {{ color: #f0f6fc; border-bottom: 1px solid #30363d; padding-bottom: 10px; margin-top: 0; }}
            h3 {{ padding-top: 10px; padding-bottom: 8px; margin-bottom: 10px; font-size: 1.1em; }}
            hr {{ border: 0; height: 1px; background-color: #30363d; margin: 20px 0; }}
            .controls-container {{ padding-bottom: 15px; margin-bottom: 15px; border-bottom: 1px solid #30363d; }}
            .filter-actions {{ display: flex; justify-content: space-between; margin-top: 10px; }}
            .filter-button {{ background-color: #21262d; border: 1px solid #30363d; color: #c9d1d9; padding: 5px 10px; border-radius: 6px; cursor: pointer; font-size: 12px; }}
            .filter-button:hover {{ background-color: #30363d; }}
            .color-swatch {{ width: 12px; height: 12px; border: 1px solid #555; border-radius: 3px; margin-right: 8px; display: inline-block; vertical-align: middle; }}
            .info-block {{ margin-bottom: 8px; font-size: 14px; line-height: 1.5; }}
            .info-block strong {{ color: #8b949e; display: inline-block; width: 180px; vertical-align: top; }}
            .tag {{ display: inline-block; background-color: #21262d; color: #c9d1d9; padding: 4px 10px; margin: 2px; border-radius: 15px; font-size: 12px; border: 1px solid #30363d; }}
            .tag-empty {{ font-style: italic; color: #8b949e; }}
            a {{ color: #58a6ff; text-decoration: none; }}
            .choices {{ margin-bottom: 15px; }}
            .choices__inner {{ background-color: #0d1117; border-radius: 6px; border: 1px solid #30363d;}}
            .choices__list--multiple .choices__item {{ background-color: #0969da; border-color: #30363d; }}
            .choices__list--dropdown {{ background-color: #161b22; border-color: #30363d; }}
            .choices__placeholder {{ color: #8b949e; }}
        </style>
        </head>
        <body>
        <div id="cy"></div>
        <div id="sidebar">
            <h2>Detalhes do Verbete</h2>
            <div id="details">Clique em um nó para ver detalhes.</div>
            <div class="controls-container">
                <h3>Filtros</h3>
                <div>
                    <label for="community-filter" style="font-size:14px; color:#8b949e; margin-bottom:5px; display:block;">Filtrar por Comunidades:</label>
                    <select id="community-filter" multiple></select>
                </div>
                <div>
                    <label for="category-filter" style="font-size:14px; color:#8b949e; margin-bottom:5px; display:block;">Filtrar por Categorias:</label>
                    <select id="category-filter" multiple></select>
                </div>
                <div class="filter-actions">
                    <button id="select-all-communities" class="filter-button">Selecionar Todas</button>
                    <button id="clear-all-filters" class="filter-button">Limpar Filtros</button>
                </div>
            </div>
        </div>
        <script>
            const verbetes = {id_to_verbete_json};
            const communityData = {community_data_json};
            const categoryData = {categories_json};

            function formatISODate(isoString) {{
                if (!isoString || isoString.includes('Não encontrado') || isoString.includes('Erro')) return 'N/A';
                try {{ return new Date(isoString).toLocaleString('pt-BR', {{ dateStyle: 'short', timeStyle: 'short' }}); }} catch (e) {{ return 'Data inválida'; }}
            }}
            function formatNumber(num) {{
                if (typeof num !== 'number') return 'N/A';
                return num.toFixed(6);
            }}
            function createTagList(dataArray, prefix = '') {{
                if (!dataArray || dataArray.length === 0) return '<span class="tag-empty">Nenhuma</span>';
                return dataArray.map(item => `<span class="tag">${{prefix}}${{item.trim()}}</span>`).join('');
            }}
            function createObjectTagList(dataObject) {{
                if (!dataObject || Object.keys(dataObject).length === 0) return '<span class="tag-empty">Nenhum</span>';
                return Object.entries(dataObject).map(([key, value]) => `<span class="tag">${{key}}: ${{value}}</span>`).join('');
            }}

            const layoutOptions = {{ name: 'preset', padding: 50, fit: true }};

            const cy = cytoscape({{
              container: document.getElementById('cy'),
              elements: {{ nodes: {nodes_json}, edges: {edges_json} }},
              style: [
                {{ selector: 'node', style: {{
                    'label': 'data(label)', 'width': 'data(size)', 'height': 'data(size)',
                    'background-color': 'data(color)', 'color': '#000000', 
                    'font-size': '12px', 'text-valign': 'center', 'text-halign': 'center', 
                    'text-wrap': 'wrap', 'text-max-width': '100px',
                    'border-color': '#000000', 'border-width': '1px', 'display': 'element'
                }}}},
                {{ selector: 'edge', style: {{
                    'width': 1.5, 'line-color': '#ffffff', 'opacity': 0.3,
                    'curve-style': 'bezier', 'display': 'element'
                }}}},
                {{ selector: 'node:selected', style: {{
                    'border-color': '#FFFF00', 'border-width': 6, 'color': '#000000',
                    'text-outline-color': '#FFFF00', 'text-outline-width': 1
                }}}},
                {{ selector: '.faded', style: {{ 'opacity': 0.1, 'text-opacity': 0 }} }}
              ],
              layout: layoutOptions
            }});

            const communityFilter = new Choices('#community-filter', {{
                removeItemButton: true, placeholder: true, placeholderValue: 'Filtrar por comunidades...', allowHTML: true
            }});
            const categoryFilter = new Choices('#category-filter', {{
                removeItemButton: true, placeholder: true, placeholderValue: 'Filtrar por categorias...'
            }});

            function populateFilters() {{
                const communityChoices = communityData
                    .sort((a, b) => (a.id === -1) - (b.id === -1) || a.id - b.id)
                    .map(c => {{
                        const label = c.name ? `${{c.name}} (${{c.size}})` : `Comunidade ${{c.id}} (${{c.size}})`;
                        return {{ value: c.id, label: `<span class="color-swatch" style="background-color:${{c.color}};"></span> ${{label}}` }};
                    }});
                communityFilter.setChoices(communityChoices, 'value', 'label', false);

                const categoryChoices = categoryData.map(cat => ({{ value: cat, label: cat }}));
                categoryFilter.setChoices(categoryChoices, 'value', 'label', false);
            }}

            function applyFilters() {{
                const selectedCommunities = communityFilter.getValue(true);
                const selectedCategories = categoryFilter.getValue(true);

                if (selectedCommunities.length === 0 && selectedCategories.length === 0) {{
                    cy.elements().style('display', 'element');
                    return;
                }}
                let nodesToShow = cy.nodes();
                if (selectedCommunities.length > 0) {{
                    const communitySelector = selectedCommunities.map(id => `[community_id = ${{id}}]`).join(', ');
                    nodesToShow = nodesToShow.filter(communitySelector);
                }}
                if (selectedCategories.length > 0) {{
                    const categorySelector = selectedCategories.map(cat => `[categories_str *= "|${{cat}}|"]`).join(', ');
                    nodesToShow = nodesToShow.filter(categorySelector);
                }}
                cy.elements().style('display', 'none');
                nodesToShow.union(nodesToShow.connectedEdges()).style('display', 'element');
            }}

            document.getElementById('community-filter').addEventListener('change', applyFilters);
            document.getElementById('category-filter').addEventListener('change', applyFilters);
            populateFilters();

            document.getElementById('select-all-communities').addEventListener('click', () => {{
                const allCommunityValues = communityData.map(c => String(c.id)); 
                communityFilter.setValue(allCommunityValues);
            }});
            document.getElementById('clear-all-filters').addEventListener('click', () => {{
                communityFilter.removeActiveItems();
                categoryFilter.removeActiveItems();
            }});
            
            cy.on('tap', 'node', function(evt) {{
                const node = evt.target;
                const data = verbetes[node.id()];
                let detailsHTML = `
                    <h3>Informações Gerais</h3>
                    <div class="info-block"><strong>Título:</strong> ${{data.titulo || 'N/A'}}</div>
                    <div class="info-block"><strong>Link:</strong> <a href="${{data.link}}" target="_blank">Abrir na Wiki</a></div>
                    <div class="info-block"><strong>ID da Comunidade:</strong> ${{data.community_id !== -1 ? data.community_id : 'N/A'}}</div>
                    <hr>
                    <h3>Métricas de Rede</h3>
                    <div class="info-block"><strong>PageRank:</strong> ${{formatNumber(data.pagerank)}}</div>
                    <div class="info-block"><strong>Betweenness Centrality:</strong> ${{formatNumber(data.betweenness_centrality)}}</div>
                    <div class="info-block"><strong>Closeness Centrality:</strong> ${{formatNumber(data.closeness_centrality)}}</div>
                    <div class="info-block"><strong>Total Degree:</strong> ${{data.total_degree || 0}}</div>
                    <hr>
                    <h3>Dados de Conteúdo</h3>
                    <div class="info-block"><strong>Categorias:</strong><br>${{createTagList(data.categorias, 'Cat: ')}}</div>
                `;
                document.getElementById('details').innerHTML = detailsHTML;
                cy.elements().addClass('faded');
                node.neighborhood().union(node).removeClass('faded');
            }});

            cy.on('tap', function(evt) {{
              if (evt.target === cy) {{
                cy.elements().removeClass('faded');
                document.getElementById('details').innerHTML = "Clique em um nó para ver detalhes";
              }}
            }});
        </script>
        </body></html>
    """

    # --- 5. SALVANDO O ARQUIVO HTML FINAL ---
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_html_path = OUTPUT_DIR / f'grafo_degree_percentil_{ts}.html'
    
    with open(output_html_path, 'w', encoding='utf-8') as f:
        f.write(html_template)

    print(f"✅ Visualização gerada com sucesso em '{output_html_path}'!")

if __name__ == '__main__':
    gerar_visualizacao_por_degree_percentil()


Carregando dados enriquecidos de '..\dados\dados_com_constraint_novo.json'...
Preparando dados para o Cytoscape com coloração por percentil de Degree Centrality...
✅ Visualização gerada com sucesso em '..\public\grafo_degree_percentil_20251016_025026.html'!


In [5]:
# gerar_visualizacao_edicoes_percentil.py
#
# Versão que colore os nós do grafo com base em PERCENTIS da métrica 'quantidade_edicoes',
# usando uma escala de cores categórica para destacar os verbetes mais ativos.

import json
import os
from datetime import datetime
from pathlib import Path
import math

def gerar_visualizacao_por_edicoes_percentil():
    """
    Gera o arquivo HTML da visualização com cores baseadas em percentis
    da quantidade de edições de cada verbete.
    """

    # --- 1. CONFIGURAÇÃO DE ARQUIVOS ---
    INPUT_DIR = Path('../dados')
    INPUT_FILENAME = 'dados_com_constraint_novo.json'
    OUTPUT_DIR = Path('../public')
    OUTPUT_DIR.mkdir(exist_ok=True)
    
    input_path = INPUT_DIR / INPUT_FILENAME

    if not input_path.exists():
        print(f"ERRO: O arquivo de dados enriquecidos não foi encontrado em '{input_path}'")
        return

    # --- 2. CARREGAMENTO DOS DADOS ENRIQUECIDOS ---
    print(f"Carregando dados enriquecidos de '{input_path}'...")
    with open(input_path, 'r', encoding='utf-8') as f:
        dados = json.load(f)
    
    verbetes_enriquecidos = dados.get('verbetes_completo', [])
    if not verbetes_enriquecidos:
        print("ERRO: Nenhum verbete encontrado no arquivo de entrada.")
        return
        
    id_to_verbete = {str(v['id']): v for v in verbetes_enriquecidos}
    titulos_ids = {v['titulo']: v['id'] for v in verbetes_enriquecidos}
    
    # --- 3. PREPARAÇÃO DOS DADOS PARA O CYTOSCAPE ---
    print("Preparando dados para o Cytoscape com coloração por percentil de Quantidade de Edições...")

    # --- ATUALIZAÇÃO: Lógica de Escala por Percentil para 'quantidade_edicoes' ---
    # 1. Ordena os verbetes por quantidade_edicoes (do maior para o menor)
    verbetes_ordenados = sorted(
        verbetes_enriquecidos, 
        key=lambda v: v.get('quantidade_edicoes', 0), 
        reverse=True
    )
    
    # 2. Calcula os pontos de corte dos percentis
    total_verbetes = len(verbetes_ordenados)
    idx_top_1_percent = int(total_verbetes * 0.01)
    idx_top_10_percent = int(total_verbetes * 0.10)
    idx_top_50_percent = int(total_verbetes * 0.50)

    # 3. Cria um mapa de ID para cor com base na posição ordenada
    id_to_color = {}
    for i, verbete in enumerate(verbetes_ordenados):
        verbete_id = str(verbete['id'])
        if i < idx_top_1_percent:
            id_to_color[verbete_id] = '#FF0000'  # Vermelho
        elif i < idx_top_10_percent:
            id_to_color[verbete_id] = '#FFFF00'  # Amarelo
        elif i < idx_top_50_percent:
            id_to_color[verbete_id] = '#FFFFFF'  # Branco
        else:
            id_to_color[verbete_id] = "#00B1EC"  # Azul Claro

    all_categories = set()
    nodes = []
    for verb in verbetes_enriquecidos:
        size = (20 + (verb.get('pagerank', 0) * 60000))
        verb_id = str(verb['id'])
        
        # ATUALIZAÇÃO: Atribui a cor a partir do mapa de percentis
        node_color = id_to_color.get(verb_id, '#808080') # Cinza como fallback

        clean_cats = []
        if 'categorias' in verb and verb['categorias']:
            for cat in verb['categorias']:
                clean_cat = cat.replace('Categoria:', '').strip()
                if clean_cat:
                    all_categories.add(clean_cat)
                    clean_cats.append(clean_cat)

        nodes.append({
            'data': {
                'id': verb_id,
                'label': verb['titulo'],
                'size': size,
                'color': node_color, # Usa a nova cor categórica
                'community_id': verb.get('community_id', -1),
                'categories_str': '|' + '|'.join(clean_cats) + '|'
            },
            'position': verb.get('position')
        })

    edges = []
    for verb in verbetes_enriquecidos:
        source_vid = str(verb['id'])
        for ref_titulo in verb.get('referencias', []):
            if ref_titulo in titulos_ids:
                target_vid = str(titulos_ids[ref_titulo])
                edges.append({'data': {'source': source_vid, 'target': target_vid}})

    community_map = {}
    for verb in verbetes_enriquecidos:
        cid = verb.get('community_id', -1)
        if cid not in community_map:
            community_map[cid] = {'id': cid, 'color': verb.get('community_color', '#6A737D'), 'size': 0}
        community_map[cid]['size'] += 1
    
    if -1 in community_map:
        community_map[-1]['name'] = 'Isolados / Outros'
    
    community_data = list(community_map.values())
    sorted_categories = sorted(list(all_categories))

    # --- 4. GERAÇÃO DO CÓDIGO HTML E JAVASCRIPT ---
    id_to_verbete_json = json.dumps(id_to_verbete, ensure_ascii=False)
    nodes_json = json.dumps(nodes, ensure_ascii=False)
    edges_json = json.dumps(edges, ensure_ascii=False)
    community_data_json = json.dumps(community_data, ensure_ascii=False)
    categories_json = json.dumps(sorted_categories, ensure_ascii=False)
    
    html_template = f"""
        <!DOCTYPE html><html>
        <head>
        <meta charset="utf-8"><title>Grafo Wikifavelas - Cor por Percentil de Edições</title>
        <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/choices.js/public/assets/styles/choices.min.css"/>
        <script src="https://cdn.jsdelivr.net/npm/choices.js/public/assets/scripts/choices.min.js"></script>
        <script src="https://unpkg.com/cytoscape@3.24.0/dist/cytoscape.min.js"></script>
        <style>
            body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Helvetica, Arial, sans-serif; margin: 0; display: flex; height: 100vh; background-color: #0d1117; color: #c9d1d9; }}
            #cy {{ position: relative; width: 70%; height: 100%; background-color: #0d1117; }}
            #sidebar {{ width: 30%; padding: 20px; background-color: #161b22; overflow-y: auto; border-left: 1px solid #30363d; box-sizing: border-box; }}
            h2, h3 {{ color: #f0f6fc; border-bottom: 1px solid #30363d; padding-bottom: 10px; margin-top: 0; }}
            h3 {{ padding-top: 10px; padding-bottom: 8px; margin-bottom: 10px; font-size: 1.1em; }}
            hr {{ border: 0; height: 1px; background-color: #30363d; margin: 20px 0; }}
            .controls-container {{ padding-bottom: 15px; margin-bottom: 15px; border-bottom: 1px solid #30363d; }}
            .filter-actions {{ display: flex; justify-content: space-between; margin-top: 10px; }}
            .filter-button {{ background-color: #21262d; border: 1px solid #30363d; color: #c9d1d9; padding: 5px 10px; border-radius: 6px; cursor: pointer; font-size: 12px; }}
            .filter-button:hover {{ background-color: #30363d; }}
            .color-swatch {{ width: 12px; height: 12px; border: 1px solid #555; border-radius: 3px; margin-right: 8px; display: inline-block; vertical-align: middle; }}
            .info-block {{ margin-bottom: 8px; font-size: 14px; line-height: 1.5; }}
            .info-block strong {{ color: #8b949e; display: inline-block; width: 180px; vertical-align: top; }}
            .tag {{ display: inline-block; background-color: #21262d; color: #c9d1d9; padding: 4px 10px; margin: 2px; border-radius: 15px; font-size: 12px; border: 1px solid #30363d; }}
            .tag-empty {{ font-style: italic; color: #8b949e; }}
            a {{ color: #58a6ff; text-decoration: none; }}
            .choices {{ margin-bottom: 15px; }}
            .choices__inner {{ background-color: #0d1117; border-radius: 6px; border: 1px solid #30363d;}}
            .choices__list--multiple .choices__item {{ background-color: #0969da; border-color: #30363d; }}
            .choices__list--dropdown {{ background-color: #161b22; border-color: #30363d; }}
            .choices__placeholder {{ color: #8b949e; }}
        </style>
        </head>
        <body>
        <div id="cy"></div>
        <div id="sidebar">
            <h2>Detalhes do Verbete</h2>
            <div id="details">Clique em um nó para ver detalhes.</div>
            <div class="controls-container">
                <h3>Filtros</h3>
                <div>
                    <label for="community-filter" style="font-size:14px; color:#8b949e; margin-bottom:5px; display:block;">Filtrar por Comunidades:</label>
                    <select id="community-filter" multiple></select>
                </div>
                <div>
                    <label for="category-filter" style="font-size:14px; color:#8b949e; margin-bottom:5px; display:block;">Filtrar por Categorias:</label>
                    <select id="category-filter" multiple></select>
                </div>
                <div class="filter-actions">
                    <button id="select-all-communities" class="filter-button">Selecionar Todas</button>
                    <button id="clear-all-filters" class="filter-button">Limpar Filtros</button>
                </div>
            </div>
        </div>
        <script>
            const verbetes = {id_to_verbete_json};
            const communityData = {community_data_json};
            const categoryData = {categories_json};

            function formatISODate(isoString) {{
                if (!isoString || isoString.includes('Não encontrado') || isoString.includes('Erro')) return 'N/A';
                try {{ return new Date(isoString).toLocaleString('pt-BR', {{ dateStyle: 'short', timeStyle: 'short' }}); }} catch (e) {{ return 'Data inválida'; }}
            }}
            function formatNumber(num) {{
                if (typeof num !== 'number') return 'N/A';
                return num.toFixed(6);
            }}
            function createTagList(dataArray, prefix = '') {{
                if (!dataArray || dataArray.length === 0) return '<span class="tag-empty">Nenhuma</span>';
                return dataArray.map(item => `<span class="tag">${{prefix}}${{item.trim()}}</span>`).join('');
            }}
            function createObjectTagList(dataObject) {{
                if (!dataObject || Object.keys(dataObject).length === 0) return '<span class="tag-empty">Nenhum</span>';
                return Object.entries(dataObject).map(([key, value]) => `<span class="tag">${{key}}: ${{value}}</span>`).join('');
            }}

            const layoutOptions = {{ name: 'preset', padding: 50, fit: true }};

            const cy = cytoscape({{
              container: document.getElementById('cy'),
              elements: {{ nodes: {nodes_json}, edges: {edges_json} }},
              style: [
                {{ selector: 'node', style: {{
                    'label': 'data(label)', 'width': 'data(size)', 'height': 'data(size)',
                    'background-color': 'data(color)', 'color': '#000000', 
                    'font-size': '12px', 'text-valign': 'center', 'text-halign': 'center', 
                    'text-wrap': 'wrap', 'text-max-width': '100px',
                    'border-color': '#000000', 'border-width': '1px', 'display': 'element'
                }}}},
                {{ selector: 'edge', style: {{
                    'width': 1.5, 'line-color': '#ffffff', 'opacity': 0.3,
                    'curve-style': 'bezier', 'display': 'element'
                }}}},
                {{ selector: 'node:selected', style: {{
                    'border-color': '#FFFF00', 'border-width': 6, 'color': '#000000',
                    'text-outline-color': '#FFFF00', 'text-outline-width': 1
                }}}},
                {{ selector: '.faded', style: {{ 'opacity': 0.1, 'text-opacity': 0 }} }}
              ],
              layout: layoutOptions
            }});

            const communityFilter = new Choices('#community-filter', {{
                removeItemButton: true, placeholder: true, placeholderValue: 'Filtrar por comunidades...', allowHTML: true
            }});
            const categoryFilter = new Choices('#category-filter', {{
                removeItemButton: true, placeholder: true, placeholderValue: 'Filtrar por categorias...'
            }});

            function populateFilters() {{
                const communityChoices = communityData
                    .sort((a, b) => (a.id === -1) - (b.id === -1) || a.id - b.id)
                    .map(c => {{
                        const label = c.name ? `${{c.name}} (${{c.size}})` : `Comunidade ${{c.id}} (${{c.size}})`;
                        return {{ value: c.id, label: `<span class="color-swatch" style="background-color:${{c.color}};"></span> ${{label}}` }};
                    }});
                communityFilter.setChoices(communityChoices, 'value', 'label', false);

                const categoryChoices = categoryData.map(cat => ({{ value: cat, label: cat }}));
                categoryFilter.setChoices(categoryChoices, 'value', 'label', false);
            }}

            function applyFilters() {{
                const selectedCommunities = communityFilter.getValue(true);
                const selectedCategories = categoryFilter.getValue(true);

                if (selectedCommunities.length === 0 && selectedCategories.length === 0) {{
                    cy.elements().style('display', 'element');
                    return;
                }}
                let nodesToShow = cy.nodes();
                if (selectedCommunities.length > 0) {{
                    const communitySelector = selectedCommunities.map(id => `[community_id = ${{id}}]`).join(', ');
                    nodesToShow = nodesToShow.filter(communitySelector);
                }}
                if (selectedCategories.length > 0) {{
                    const categorySelector = selectedCategories.map(cat => `[categories_str *= "|${{cat}}|"]`).join(', ');
                    nodesToShow = nodesToShow.filter(categorySelector);
                }}
                cy.elements().style('display', 'none');
                nodesToShow.union(nodesToShow.connectedEdges()).style('display', 'element');
            }}

            document.getElementById('community-filter').addEventListener('change', applyFilters);
            document.getElementById('category-filter').addEventListener('change', applyFilters);
            populateFilters();

            document.getElementById('select-all-communities').addEventListener('click', () => {{
                const allCommunityValues = communityData.map(c => String(c.id)); 
                communityFilter.setValue(allCommunityValues);
            }});
            document.getElementById('clear-all-filters').addEventListener('click', () => {{
                communityFilter.removeActiveItems();
                categoryFilter.removeActiveItems();
            }});
            
            cy.on('tap', 'node', function(evt) {{
                const node = evt.target;
                const data = verbetes[node.id()];
                let detailsHTML = `
                    <h3>Informações Gerais</h3>
                    <div class="info-block"><strong>Título:</strong> ${{data.titulo || 'N/A'}}</div>
                    <div class="info-block"><strong>Link:</strong> <a href="${{data.link}}" target="_blank">Abrir na Wiki</a></div>
                    <div class="info-block"><strong>ID da Comunidade:</strong> ${{data.community_id !== -1 ? data.community_id : 'N/A'}}</div>
                    <div class="info-block"><strong>Qtde. Edições:</strong> ${{data.quantidade_edicoes || 0}}</div>
                    <hr>
                    <h3>Métricas de Rede</h3>
                    <div class="info-block"><strong>PageRank:</strong> ${{formatNumber(data.pagerank)}}</div>
                    <div class="info-block"><strong>Betweenness Centrality:</strong> ${{formatNumber(data.betweenness_centrality)}}</div>
                    <div class="info-block"><strong>Closeness Centrality:</strong> ${{formatNumber(data.closeness_centrality)}}</div>
                    <div class="info-block"><strong>Total Degree:</strong> ${{data.total_degree || 0}}</div>
                    <hr>
                    <h3>Dados de Conteúdo</h3>
                    <div class="info-block"><strong>Categorias:</strong><br>${{createTagList(data.categorias, 'Cat: ')}}</div>
                `;
                document.getElementById('details').innerHTML = detailsHTML;
                cy.elements().addClass('faded');
                node.neighborhood().union(node).removeClass('faded');
            }});

            cy.on('tap', function(evt) {{
              if (evt.target === cy) {{
                cy.elements().removeClass('faded');
                document.getElementById('details').innerHTML = "Clique em um nó para ver detalhes";
              }}
            }});
        </script>
        </body></html>
    """

    # --- 5. SALVANDO O ARQUIVO HTML FINAL ---
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_html_path = OUTPUT_DIR / f'grafo_edicoes_percentil_{ts}.html'
    
    with open(output_html_path, 'w', encoding='utf-8') as f:
        f.write(html_template)

    print(f"✅ Visualização gerada com sucesso em '{output_html_path}'!")

if __name__ == '__main__':
    gerar_visualizacao_por_edicoes_percentil()


Carregando dados enriquecidos de '..\dados\dados_com_constraint_novo.json'...
Preparando dados para o Cytoscape com coloração por percentil de Quantidade de Edições...
✅ Visualização gerada com sucesso em '..\public\grafo_edicoes_percentil_20251016_025040.html'!


In [6]:
# gerar_visualizacao_composta_percentil.py
#
# Versão final que colore os nós do grafo com base em PERCENTIS da
# métrica composta personalizada, destacando os "super-nós" da rede.

import json
import os
from datetime import datetime
from pathlib import Path
import math

def gerar_visualizacao_por_metrica_composta():
    """
    Gera o arquivo HTML da visualização com cores baseadas em percentis
    da métrica composta.
    """

    # --- 1. CONFIGURAÇÃO DE ARQUIVOS ---
    INPUT_DIR = Path('../dados')
    # Lê o arquivo gerado pelo script 'calcular_metrica_composta.py'
    INPUT_FILENAME = 'dados_com_constraint_novo.json'
    OUTPUT_DIR = Path('../public')
    OUTPUT_DIR.mkdir(exist_ok=True)
    
    input_path = INPUT_DIR / INPUT_FILENAME

    if not input_path.exists():
        print(f"ERRO: O arquivo de dados enriquecidos não foi encontrado em '{input_path}'")
        return

    # --- 2. CARREGAMENTO DOS DADOS ENRIQUECIDOS ---
    print(f"Carregando dados enriquecidos de '{input_path}'...")
    with open(input_path, 'r', encoding='utf-8') as f:
        dados = json.load(f)
    
    verbetes_enriquecidos = dados.get('verbetes_completo', [])
    if not verbetes_enriquecidos:
        print("ERRO: Nenhum verbete encontrado no arquivo de entrada.")
        return
        
    id_to_verbete = {str(v['id']): v for v in verbetes_enriquecidos}
    titulos_ids = {v['titulo']: v['id'] for v in verbetes_enriquecidos}
    
    # --- 3. PREPARAÇÃO DOS DADOS PARA O CYTOSCAPE ---
    print("Preparando dados para o Cytoscape com coloração por percentil da Métrica Composta...")

    # --- Lógica de Escala por Percentil para 'metrica_composta' ---
    # 1. Ordena os verbetes pela métrica composta (o arquivo já vem ordenado, mas garantimos aqui)
    verbetes_ordenados = sorted(
        verbetes_enriquecidos, 
        key=lambda v: v.get('metrica_composta', 0), 
        reverse=True
    )
    
    # 2. Calcula os pontos de corte dos percentis
    total_verbetes = len(verbetes_ordenados)
    idx_top_1_percent = int(total_verbetes * 0.01)
    idx_top_10_percent = int(total_verbetes * 0.10)
    idx_top_50_percent = int(total_verbetes * 0.50)

    # 3. Cria um mapa de ID para cor com base na posição ordenada
    id_to_color = {}
    for i, verbete in enumerate(verbetes_ordenados):
        verbete_id = str(verbete['id'])
        if i < idx_top_1_percent:
            id_to_color[verbete_id] = '#FF0000'  # Vermelho
        elif i < idx_top_10_percent:
            id_to_color[verbete_id] = '#FFFF00'  # Amarelo
        elif i < idx_top_50_percent:
            id_to_color[verbete_id] = '#FFFFFF'  # Branco
        else:
            id_to_color[verbete_id] = '#ADD8E6'  # Azul Claro

    all_categories = set()
    nodes = []
    for verb in verbetes_enriquecidos:
        size = (20 + (verb.get('pagerank', 0) * 60000))
        verb_id = str(verb['id'])
        
        # Atribui a cor a partir do mapa de percentis
        node_color = id_to_color.get(verb_id, '#808080') # Cinza como fallback

        clean_cats = []
        if 'categorias' in verb and verb['categorias']:
            for cat in verb['categorias']:
                clean_cat = cat.replace('Categoria:', '').strip()
                if clean_cat:
                    all_categories.add(clean_cat)
                    clean_cats.append(clean_cat)

        nodes.append({
            'data': {
                'id': verb_id,
                'label': verb['titulo'],
                'size': size,
                'color': node_color,
                'community_id': verb.get('community_id', -1),
                'categories_str': '|' + '|'.join(clean_cats) + '|'
            },
            'position': verb.get('position')
        })

    edges = []
    for verb in verbetes_enriquecidos:
        source_vid = str(verb['id'])
        for ref_titulo in verb.get('referencias', []):
            if ref_titulo in titulos_ids:
                target_vid = str(titulos_ids[ref_titulo])
                edges.append({'data': {'source': source_vid, 'target': target_vid}})

    community_map = {}
    for verb in verbetes_enriquecidos:
        cid = verb.get('community_id', -1)
        if cid not in community_map:
            community_map[cid] = {'id': cid, 'color': verb.get('community_color', '#6A737D'), 'size': 0}
        community_map[cid]['size'] += 1
    
    if -1 in community_map:
        community_map[-1]['name'] = 'Isolados / Outros'
    
    community_data = list(community_map.values())
    sorted_categories = sorted(list(all_categories))

    # --- 4. GERAÇÃO DO CÓDIGO HTML E JAVASCRIPT ---
    id_to_verbete_json = json.dumps(id_to_verbete, ensure_ascii=False)
    nodes_json = json.dumps(nodes, ensure_ascii=False)
    edges_json = json.dumps(edges, ensure_ascii=False)
    community_data_json = json.dumps(community_data, ensure_ascii=False)
    categories_json = json.dumps(sorted_categories, ensure_ascii=False)
    
    html_template = f"""
        <!DOCTYPE html><html>
        <head>
        <meta charset="utf-8"><title>Grafo Wikifavelas - Cor por Métrica Composta</title>
        <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/choices.js/public/assets/styles/choices.min.css"/>
        <script src="https://cdn.jsdelivr.net/npm/choices.js/public/assets/scripts/choices.min.js"></script>
        <script src="https://unpkg.com/cytoscape@3.24.0/dist/cytoscape.min.js"></script>
        <style>
            body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Helvetica, Arial, sans-serif; margin: 0; display: flex; height: 100vh; background-color: #0d1117; color: #c9d1d9; }}
            #cy {{ position: relative; width: 70%; height: 100%; background-color: #0d1117; }}
            #sidebar {{ width: 30%; padding: 20px; background-color: #161b22; overflow-y: auto; border-left: 1px solid #30363d; box-sizing: border-box; }}
            h2, h3 {{ color: #f0f6fc; border-bottom: 1px solid #30363d; padding-bottom: 10px; margin-top: 0; }}
            h3 {{ padding-top: 10px; padding-bottom: 8px; margin-bottom: 10px; font-size: 1.1em; }}
            hr {{ border: 0; height: 1px; background-color: #30363d; margin: 20px 0; }}
            .controls-container {{ padding-bottom: 15px; margin-bottom: 15px; border-bottom: 1px solid #30363d; }}
            .filter-actions {{ display: flex; justify-content: space-between; margin-top: 10px; }}
            .filter-button {{ background-color: #21262d; border: 1px solid #30363d; color: #c9d1d9; padding: 5px 10px; border-radius: 6px; cursor: pointer; font-size: 12px; }}
            .filter-button:hover {{ background-color: #30363d; }}
            .color-swatch {{ width: 12px; height: 12px; border: 1px solid #555; border-radius: 3px; margin-right: 8px; display: inline-block; vertical-align: middle; }}
            .info-block {{ margin-bottom: 8px; font-size: 14px; line-height: 1.5; }}
            .info-block strong {{ color: #8b949e; display: inline-block; width: 180px; vertical-align: top; }}
            .tag {{ display: inline-block; background-color: #21262d; color: #c9d1d9; padding: 4px 10px; margin: 2px; border-radius: 15px; font-size: 12px; border: 1px solid #30363d; }}
            .tag-empty {{ font-style: italic; color: #8b949e; }}
            a {{ color: #58a6ff; text-decoration: none; }}
            .choices {{ margin-bottom: 15px; }}
            .choices__inner {{ background-color: #0d1117; border-radius: 6px; border: 1px solid #30363d;}}
            .choices__list--multiple .choices__item {{ background-color: #0969da; border-color: #30363d; }}
            .choices__list--dropdown {{ background-color: #161b22; border-color: #30363d; }}
            .choices__placeholder {{ color: #8b949e; }}
        </style>
        </head>
        <body>
        <div id="cy"></div>
        <div id="sidebar">
            <h2>Detalhes do Verbete</h2>
            <div id="details">Clique em um nó para ver detalhes.</div>
            <div class="controls-container">
                <h3>Filtros</h3>
                <div>
                    <label for="community-filter" style="font-size:14px; color:#8b949e; margin-bottom:5px; display:block;">Filtrar por Comunidades:</label>
                    <select id="community-filter" multiple></select>
                </div>
                <div>
                    <label for="category-filter" style="font-size:14px; color:#8b949e; margin-bottom:5px; display:block;">Filtrar por Categorias:</label>
                    <select id="category-filter" multiple></select>
                </div>
                <div class="filter-actions">
                    <button id="select-all-communities" class="filter-button">Selecionar Todas</button>
                    <button id="clear-all-filters" class="filter-button">Limpar Filtros</button>
                </div>
            </div>
        </div>
        <script>
            const verbetes = {id_to_verbete_json};
            const communityData = {community_data_json};
            const categoryData = {categories_json};

            function formatISODate(isoString) {{
                if (!isoString || isoString.includes('Não encontrado') || isoString.includes('Erro')) return 'N/A';
                try {{ return new Date(isoString).toLocaleString('pt-BR', {{ dateStyle: 'short', timeStyle: 'short' }}); }} catch (e) {{ return 'Data inválida'; }}
            }}
            function formatNumber(num) {{
                if (typeof num !== 'number') return 'N/A';
                return num.toFixed(6);
            }}
            function createTagList(dataArray, prefix = '') {{
                if (!dataArray || dataArray.length === 0) return '<span class="tag-empty">Nenhuma</span>';
                return dataArray.map(item => `<span class="tag">${{prefix}}${{item.trim()}}</span>`).join('');
            }}
            function createObjectTagList(dataObject) {{
                if (!dataObject || Object.keys(dataObject).length === 0) return '<span class="tag-empty">Nenhum</span>';
                return Object.entries(dataObject).map(([key, value]) => `<span class="tag">${{key}}: ${{value}}</span>`).join('');
            }}

            const layoutOptions = {{ name: 'preset', padding: 50, fit: true }};

            const cy = cytoscape({{
              container: document.getElementById('cy'),
              elements: {{ nodes: {nodes_json}, edges: {edges_json} }},
              style: [
                {{ selector: 'node', style: {{
                    'label': 'data(label)', 'width': 'data(size)', 'height': 'data(size)',
                    'background-color': 'data(color)', 'color': '#000000', 
                    'font-size': '12px', 'text-valign': 'center', 'text-halign': 'center', 
                    'text-wrap': 'wrap', 'text-max-width': '100px',
                    'border-color': '#000000', 'border-width': '1px', 'display': 'element'
                }}}},
                {{ selector: 'edge', style: {{
                    'width': 1.5, 'line-color': '#ffffff', 'opacity': 0.3,
                    'curve-style': 'bezier', 'display': 'element'
                }}}},
                {{ selector: 'node:selected', style: {{
                    'border-color': '#FFFF00', 'border-width': 6, 'color': '#000000',
                    'text-outline-color': '#FFFF00', 'text-outline-width': 1
                }}}},
                {{ selector: '.faded', style: {{ 'opacity': 0.1, 'text-opacity': 0 }} }}
              ],
              layout: layoutOptions
            }});

            const communityFilter = new Choices('#community-filter', {{
                removeItemButton: true, placeholder: true, placeholderValue: 'Filtrar por comunidades...', allowHTML: true
            }});
            const categoryFilter = new Choices('#category-filter', {{
                removeItemButton: true, placeholder: true, placeholderValue: 'Filtrar por categorias...'
            }});

            function populateFilters() {{
                const communityChoices = communityData
                    .sort((a, b) => (a.id === -1) - (b.id === -1) || a.id - b.id)
                    .map(c => {{
                        const label = c.name ? `${{c.name}} (${{c.size}})` : `Comunidade ${{c.id}} (${{c.size}})`;
                        return {{ value: c.id, label: `<span class="color-swatch" style="background-color:${{c.color}};"></span> ${{label}}` }};
                    }});
                communityFilter.setChoices(communityChoices, 'value', 'label', false);

                const categoryChoices = categoryData.map(cat => ({{ value: cat, label: cat }}));
                categoryFilter.setChoices(categoryChoices, 'value', 'label', false);
            }}

            function applyFilters() {{
                const selectedCommunities = communityFilter.getValue(true);
                const selectedCategories = categoryFilter.getValue(true);

                if (selectedCommunities.length === 0 && selectedCategories.length === 0) {{
                    cy.elements().style('display', 'element');
                    return;
                }}
                let nodesToShow = cy.nodes();
                if (selectedCommunities.length > 0) {{
                    const communitySelector = selectedCommunities.map(id => `[community_id = ${{id}}]`).join(', ');
                    nodesToShow = nodesToShow.filter(communitySelector);
                }}
                if (selectedCategories.length > 0) {{
                    const categorySelector = selectedCategories.map(cat => `[categories_str *= "|${{cat}}|"]`).join(', ');
                    nodesToShow = nodesToShow.filter(categorySelector);
                }}
                cy.elements().style('display', 'none');
                nodesToShow.union(nodesToShow.connectedEdges()).style('display', 'element');
            }}

            document.getElementById('community-filter').addEventListener('change', applyFilters);
            document.getElementById('category-filter').addEventListener('change', applyFilters);
            populateFilters();

            document.getElementById('select-all-communities').addEventListener('click', () => {{
                const allCommunityValues = communityData.map(c => String(c.id)); 
                communityFilter.setValue(allCommunityValues);
            }});
            document.getElementById('clear-all-filters').addEventListener('click', () => {{
                communityFilter.removeActiveItems();
                categoryFilter.removeActiveItems();
            }});
            
            cy.on('tap', 'node', function(evt) {{
                const node = evt.target;
                const data = verbetes[node.id()];
                let detailsHTML = `
                    <h3>Informações Gerais</h3>
                    <div class="info-block"><strong>Título:</strong> ${{data.titulo || 'N/A'}}</div>
                    <div class="info-block"><strong>Link:</strong> <a href="${{data.link}}" target="_blank">Abrir na Wiki</a></div>
                    <div class="info-block"><strong>ID da Comunidade:</strong> ${{data.community_id !== -1 ? data.community_id : 'N/A'}}</div>
                    <hr>
                    <h3>Métricas de Rede</h3>
                    <div class="info-block"><strong>Métrica Composta:</strong> ${{formatNumber(data.metrica_composta)}}</div>
                    <div class="info-block"><strong>PageRank:</strong> ${{formatNumber(data.pagerank)}}</div>
                    <div class="info-block"><strong>Betweenness:</strong> ${{formatNumber(data.betweenness_centrality)}}</div>
                    <div class="info-block"><strong>Closeness:</strong> ${{formatNumber(data.closeness_centrality)}}</div>
                    <div class="info-block"><strong>Total Degree:</strong> ${{data.total_degree || 0}}</div>
                    <div class="info-block"><strong>Qtde. Edições:</strong> ${{data.quantidade_edicoes || 0}}</div>
                    <hr>
                    <h3>Dados de Conteúdo</h3>
                    <div class="info-block"><strong>Categorias:</strong><br>${{createTagList(data.categorias, 'Cat: ')}}</div>
                `;
                document.getElementById('details').innerHTML = detailsHTML;
                cy.elements().addClass('faded');
                node.neighborhood().union(node).removeClass('faded');
            }});

            cy.on('tap', function(evt) {{
              if (evt.target === cy) {{
                cy.elements().removeClass('faded');
                document.getElementById('details').innerHTML = "Clique em um nó para ver detalhes";
              }}
            }});
        </script>
        </body></html>
    """

    # --- 5. SALVANDO O ARQUIVO HTML FINAL ---
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_html_path = OUTPUT_DIR / f'grafo_metrica_composta_percentil_{ts}.html'
    
    with open(output_html_path, 'w', encoding='utf-8') as f:
        f.write(html_template)

    print(f"✅ Visualização gerada com sucesso em '{output_html_path}'!")

if __name__ == '__main__':
    gerar_visualizacao_por_metrica_composta()


Carregando dados enriquecidos de '..\dados\dados_com_constraint_novo.json'...
Preparando dados para o Cytoscape com coloração por percentil da Métrica Composta...
✅ Visualização gerada com sucesso em '..\public\grafo_metrica_composta_percentil_20251016_025058.html'!


In [7]:
# gerar_visualizacao_constraint_percentil.py
#
# Versão que colore os nós do grafo com base em PERCENTIS da métrica 'constraint',
# usando uma escala de cores categórica para destacar os nós que atuam como
# pontes entre diferentes clusters (baixo constraint).

import json
import os
from datetime import datetime
from pathlib import Path
import math

def gerar_visualizacao_por_constraint_percentil():
    """
    Gera o arquivo HTML da visualização com cores baseadas em percentis
    da métrica de Restrição (Constraint) de Burt.
    """

    # --- 1. CONFIGURAÇÃO DE ARQUIVOS ---
    INPUT_DIR = Path('../dados')
    # Lê o arquivo que já contém a métrica de constraint
    INPUT_FILENAME = 'dados_com_constraint_novo.json'
    OUTPUT_DIR = Path('../public')
    OUTPUT_DIR.mkdir(exist_ok=True)
    
    input_path = INPUT_DIR / INPUT_FILENAME

    if not input_path.exists():
        print(f"ERRO: O arquivo de dados enriquecidos não foi encontrado em '{input_path}'")
        return

    # --- 2. CARREGAMENTO DOS DADOS ENRIQUECIDOS ---
    print(f"Carregando dados enriquecidos de '{input_path}'...")
    with open(input_path, 'r', encoding='utf-8') as f:
        dados = json.load(f)
    
    verbetes_enriquecidos = dados.get('verbetes_completo', [])
    if not verbetes_enriquecidos:
        print("ERRO: Nenhum verbete encontrado no arquivo de entrada.")
        return
        
    id_to_verbete = {str(v['id']): v for v in verbetes_enriquecidos}
    titulos_ids = {v['titulo']: v['id'] for v in verbetes_enriquecidos}
    
    # --- 3. PREPARAÇÃO DOS DADOS PARA O CYTOSCAPE ---
    print("Preparando dados para o Cytoscape com coloração por percentil de Constraint...")

    # --- ATUALIZAÇÃO: Lógica de Escala por Percentil para 'constraint' ---
    # 1. Ordena os verbetes por constraint (do MENOR para o MAIOR, pois baixo é melhor)
    verbetes_ordenados = sorted(
        verbetes_enriquecidos, 
        key=lambda v: v.get('constraint', 1.0), # Default 1.0 (alto) se a métrica faltar
        reverse=False # Importante: ordem ascendente
    )
    
    # 2. Calcula os pontos de corte dos percentis
    total_verbetes = len(verbetes_ordenados)
    idx_top_1_percent = int(total_verbetes * 0.01)
    idx_top_10_percent = int(total_verbetes * 0.10)
    idx_top_50_percent = int(total_verbetes * 0.50)

    # 3. Cria um mapa de ID para cor com base na posição ordenada
    id_to_color = {}
    for i, verbete in enumerate(verbetes_ordenados):
        verbete_id = str(verbete['id'])
        if i < idx_top_1_percent:
            id_to_color[verbete_id] = '#FF0000'  # Vermelho (os melhores "brokers")
        elif i < idx_top_10_percent:
            id_to_color[verbete_id] = '#FFFF00'  # Amarelo
        elif i < idx_top_50_percent:
            id_to_color[verbete_id] = '#FFFFFF'  # Branco
        else:
            id_to_color[verbete_id] = '#00B1EC'  # Azul Claro (os mais "restritos")

    all_categories = set()
    nodes = []
    for verb in verbetes_enriquecidos:
        size = (20 + (verb.get('pagerank', 0) * 60000))
        verb_id = str(verb['id'])
        
        # Atribui a cor a partir do mapa de percentis
        node_color = id_to_color.get(verb_id, '#808080') # Cinza como fallback

        clean_cats = []
        if 'categorias' in verb and verb['categorias']:
            for cat in verb['categorias']:
                clean_cat = cat.replace('Categoria:', '').strip()
                if clean_cat:
                    all_categories.add(clean_cat)
                    clean_cats.append(clean_cat)

        nodes.append({
            'data': {
                'id': verb_id,
                'label': verb['titulo'],
                'size': size,
                'color': node_color,
                'community_id': verb.get('community_id', -1),
                'categories_str': '|' + '|'.join(clean_cats) + '|'
            },
            'position': verb.get('position')
        })

    edges = []
    for verb in verbetes_enriquecidos:
        source_vid = str(verb['id'])
        for ref_titulo in verb.get('referencias', []):
            if ref_titulo in titulos_ids:
                target_vid = str(titulos_ids[ref_titulo])
                edges.append({'data': {'source': source_vid, 'target': target_vid}})

    community_map = {}
    for verb in verbetes_enriquecidos:
        cid = verb.get('community_id', -1)
        if cid not in community_map:
            community_map[cid] = {'id': cid, 'color': verb.get('community_color', '#6A737D'), 'size': 0}
        community_map[cid]['size'] += 1
    
    if -1 in community_map:
        community_map[-1]['name'] = 'Isolados / Outros'
    
    community_data = list(community_map.values())
    sorted_categories = sorted(list(all_categories))

    # --- 4. GERAÇÃO DO CÓDIGO HTML E JAVASCRIPT ---
    id_to_verbete_json = json.dumps(id_to_verbete, ensure_ascii=False)
    nodes_json = json.dumps(nodes, ensure_ascii=False)
    edges_json = json.dumps(edges, ensure_ascii=False)
    community_data_json = json.dumps(community_data, ensure_ascii=False)
    categories_json = json.dumps(sorted_categories, ensure_ascii=False)
    
    html_template = f"""
        <!DOCTYPE html><html>
        <head>
        <meta charset="utf-8"><title>Grafo Wikifavelas - Cor por Percentil de Constraint</title>
        <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/choices.js/public/assets/styles/choices.min.css"/>
        <script src="https://cdn.jsdelivr.net/npm/choices.js/public/assets/scripts/choices.min.js"></script>
        <script src="https://unpkg.com/cytoscape@3.24.0/dist/cytoscape.min.js"></script>
        <style>
            body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Helvetica, Arial, sans-serif; margin: 0; display: flex; height: 100vh; background-color: #0d1117; color: #c9d1d9; }}
            #cy {{ position: relative; width: 70%; height: 100%; background-color: #0d1117; }}
            #sidebar {{ width: 30%; padding: 20px; background-color: #161b22; overflow-y: auto; border-left: 1px solid #30363d; box-sizing: border-box; }}
            h2, h3 {{ color: #f0f6fc; border-bottom: 1px solid #30363d; padding-bottom: 10px; margin-top: 0; }}
            h3 {{ padding-top: 10px; padding-bottom: 8px; margin-bottom: 10px; font-size: 1.1em; }}
            hr {{ border: 0; height: 1px; background-color: #30363d; margin: 20px 0; }}
            .controls-container {{ padding-bottom: 15px; margin-bottom: 15px; border-bottom: 1px solid #30363d; }}
            .filter-actions {{ display: flex; justify-content: space-between; margin-top: 10px; }}
            .filter-button {{ background-color: #21262d; border: 1px solid #30363d; color: #c9d1d9; padding: 5px 10px; border-radius: 6px; cursor: pointer; font-size: 12px; }}
            .filter-button:hover {{ background-color: #30363d; }}
            .color-swatch {{ width: 12px; height: 12px; border: 1px solid #555; border-radius: 3px; margin-right: 8px; display: inline-block; vertical-align: middle; }}
            .info-block {{ margin-bottom: 8px; font-size: 14px; line-height: 1.5; }}
            .info-block strong {{ color: #8b949e; display: inline-block; width: 180px; vertical-align: top; }}
            .tag {{ display: inline-block; background-color: #21262d; color: #c9d1d9; padding: 4px 10px; margin: 2px; border-radius: 15px; font-size: 12px; border: 1px solid #30363d; }}
            .tag-empty {{ font-style: italic; color: #8b949e; }}
            a {{ color: #58a6ff; text-decoration: none; }}
            .choices {{ margin-bottom: 15px; }}
            .choices__inner {{ background-color: #0d1117; border-radius: 6px; border: 1px solid #30363d;}}
            .choices__list--multiple .choices__item {{ background-color: #0969da; border-color: #30363d; }}
            .choices__list--dropdown {{ background-color: #161b22; border-color: #30363d; }}
            .choices__placeholder {{ color: #8b949e; }}
        </style>
        </head>
        <body>
        <div id="cy"></div>
        <div id="sidebar">
            <h2>Detalhes do Verbete</h2>
            <div id="details">Clique em um nó para ver detalhes.</div>
            <div class="controls-container">
                <h3>Filtros</h3>
                <div>
                    <label for="community-filter" style="font-size:14px; color:#8b949e; margin-bottom:5px; display:block;">Filtrar por Comunidades:</label>
                    <select id="community-filter" multiple></select>
                </div>
                <div>
                    <label for="category-filter" style="font-size:14px; color:#8b949e; margin-bottom:5px; display:block;">Filtrar por Categorias:</label>
                    <select id="category-filter" multiple></select>
                </div>
                <div class="filter-actions">
                    <button id="select-all-communities" class="filter-button">Selecionar Todas</button>
                    <button id="clear-all-filters" class="filter-button">Limpar Filtros</button>
                </div>
            </div>
        </div>
        <script>
            const verbetes = {id_to_verbete_json};
            const communityData = {community_data_json};
            const categoryData = {categories_json};

            function formatISODate(isoString) {{
                if (!isoString || isoString.includes('Não encontrado') || isoString.includes('Erro')) return 'N/A';
                try {{ return new Date(isoString).toLocaleString('pt-BR', {{ dateStyle: 'short', timeStyle: 'short' }}); }} catch (e) {{ return 'Data inválida'; }}
            }}
            function formatNumber(num) {{
                if (typeof num !== 'number') return 'N/A';
                return num.toFixed(6);
            }}
            function createTagList(dataArray, prefix = '') {{
                if (!dataArray || dataArray.length === 0) return '<span class="tag-empty">Nenhuma</span>';
                return dataArray.map(item => `<span class="tag">${{prefix}}${{item.trim()}}</span>`).join('');
            }}
            function createObjectTagList(dataObject) {{
                if (!dataObject || Object.keys(dataObject).length === 0) return '<span class="tag-empty">Nenhum</span>';
                return Object.entries(dataObject).map(([key, value]) => `<span class="tag">${{key}}: ${{value}}</span>`).join('');
            }}

            const layoutOptions = {{ name: 'preset', padding: 50, fit: true }};

            const cy = cytoscape({{
              container: document.getElementById('cy'),
              elements: {{ nodes: {nodes_json}, edges: {edges_json} }},
              style: [
                {{ selector: 'node', style: {{
                    'label': 'data(label)', 'width': 'data(size)', 'height': 'data(size)',
                    'background-color': 'data(color)', 'color': '#000000', 
                    'font-size': '12px', 'text-valign': 'center', 'text-halign': 'center', 
                    'text-wrap': 'wrap', 'text-max-width': '100px',
                    'border-color': '#000000', 'border-width': '1px', 'display': 'element'
                }}}},
                {{ selector: 'edge', style: {{
                    'width': 1.5, 'line-color': '#ffffff', 'opacity': 0.3,
                    'curve-style': 'bezier', 'display': 'element'
                }}}},
                {{ selector: 'node:selected', style: {{
                    'border-color': '#FFFF00', 'border-width': 6, 'color': '#000000',
                    'text-outline-color': '#FFFF00', 'text-outline-width': 1
                }}}},
                {{ selector: '.faded', style: {{ 'opacity': 0.1, 'text-opacity': 0 }} }}
              ],
              layout: layoutOptions
            }});

            const communityFilter = new Choices('#community-filter', {{
                removeItemButton: true, placeholder: true, placeholderValue: 'Filtrar por comunidades...', allowHTML: true
            }});
            const categoryFilter = new Choices('#category-filter', {{
                removeItemButton: true, placeholder: true, placeholderValue: 'Filtrar por categorias...'
            }});

            function populateFilters() {{
                const communityChoices = communityData
                    .sort((a, b) => (a.id === -1) - (b.id === -1) || a.id - b.id)
                    .map(c => {{
                        const label = c.name ? `${{c.name}} (${{c.size}})` : `Comunidade ${{c.id}} (${{c.size}})`;
                        return {{ value: c.id, label: `<span class="color-swatch" style="background-color:${{c.color}};"></span> ${{label}}` }};
                    }});
                communityFilter.setChoices(communityChoices, 'value', 'label', false);

                const categoryChoices = categoryData.map(cat => ({{ value: cat, label: cat }}));
                categoryFilter.setChoices(categoryChoices, 'value', 'label', false);
            }}

            function applyFilters() {{
                const selectedCommunities = communityFilter.getValue(true);
                const selectedCategories = categoryFilter.getValue(true);

                if (selectedCommunities.length === 0 && selectedCategories.length === 0) {{
                    cy.elements().style('display', 'element');
                    return;
                }}
                let nodesToShow = cy.nodes();
                if (selectedCommunities.length > 0) {{
                    const communitySelector = selectedCommunities.map(id => `[community_id = ${{id}}]`).join(', ');
                    nodesToShow = nodesToShow.filter(communitySelector);
                }}
                if (selectedCategories.length > 0) {{
                    const categorySelector = selectedCategories.map(cat => `[categories_str *= "|${{cat}}|"]`).join(', ');
                    nodesToShow = nodesToShow.filter(categorySelector);
                }}
                cy.elements().style('display', 'none');
                nodesToShow.union(nodesToShow.connectedEdges()).style('display', 'element');
            }}

            document.getElementById('community-filter').addEventListener('change', applyFilters);
            document.getElementById('category-filter').addEventListener('change', applyFilters);
            populateFilters();

            document.getElementById('select-all-communities').addEventListener('click', () => {{
                const allCommunityValues = communityData.map(c => String(c.id)); 
                communityFilter.setValue(allCommunityValues);
            }});
            document.getElementById('clear-all-filters').addEventListener('click', () => {{
                communityFilter.removeActiveItems();
                categoryFilter.removeActiveItems();
            }});
            
            cy.on('tap', 'node', function(evt) {{
                const node = evt.target;
                const data = verbetes[node.id()];
                let detailsHTML = `
                    <h3>Informações Gerais</h3>
                    <div class="info-block"><strong>Título:</strong> ${{data.titulo || 'N/A'}}</div>
                    <div class="info-block"><strong>Link:</strong> <a href="${{data.link}}" target="_blank">Abrir na Wiki</a></div>
                    <div class="info-block"><strong>ID da Comunidade:</strong> ${{data.community_id !== -1 ? data.community_id : 'N/A'}}</div>
                    <hr>
                    <h3>Métricas de Rede</h3>
                    <div class="info-block"><strong>Constraint:</strong> ${{formatNumber(data.constraint)}}</div>
                    <div class="info-block"><strong>PageRank:</strong> ${{formatNumber(data.pagerank)}}</div>
                    <div class="info-block"><strong>Betweenness:</strong> ${{formatNumber(data.betweenness_centrality)}}</div>
                    <div class="info-block"><strong>Total Degree:</strong> ${{data.total_degree || 0}}</div>
                    <hr>
                    <h3>Dados de Conteúdo</h3>
                    <div class="info-block"><strong>Categorias:</strong><br>${{createTagList(data.categorias, 'Cat: ')}}</div>
                `;
                document.getElementById('details').innerHTML = detailsHTML;
                cy.elements().addClass('faded');
                node.neighborhood().union(node).removeClass('faded');
            }});

            cy.on('tap', function(evt) {{
              if (evt.target === cy) {{
                cy.elements().removeClass('faded');
                document.getElementById('details').innerHTML = "Clique em um nó para ver detalhes";
              }}
            }});
        </script>
        </body></html>
    """

    # --- 5. SALVANDO O ARQUIVO HTML FINAL ---
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_html_path = OUTPUT_DIR / f'grafo_constraint_percentil_{ts}.html'
    
    with open(output_html_path, 'w', encoding='utf-8') as f:
        f.write(html_template)

    print(f"✅ Visualização gerada com sucesso em '{output_html_path}'!")

if __name__ == '__main__':
    gerar_visualizacao_por_constraint_percentil()


Carregando dados enriquecidos de '..\dados\dados_com_constraint_novo.json'...
Preparando dados para o Cytoscape com coloração por percentil de Constraint...
✅ Visualização gerada com sucesso em '..\public\grafo_constraint_percentil_20251016_025113.html'!
